In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

from utils import * 
import pickle
from tqdm import tqdm 

import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

# === Visualization ===
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import FancyArrowPatch, ConnectionPatch
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.ticker import LogLocator, MaxNLocator, NullLocator
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib.path import Path
import matplotlib.gridspec as gridspec
from matplotlib.legend_handler import HandlerTuple
from matplotlib.patches import Rectangle, Circle, Wedge
from matplotlib.ticker import FixedLocator, FormatStrFormatter
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


olympic_colors = [
    "#0085C7",  # Blue
    "#F4C300",  # Yellow
    "#000000",  # Black
    "#009F3D",  # Green
    "#DF0024"   # Red
]

sns.set(
    context='paper',
    font_scale=2.2,
    font="Arial",
    palette='deep',
    style='ticks',
    color_codes=True,
    rc={
        "mathtext.fontset": "cm",
        "axes.linewidth": 2,
        "font.size": 16,
        "figure.dpi": 100,
        "text.usetex": False,
        "lines.linewidth": 1.5,
        "axes.labelpad": 0,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 6,
        "ytick.major.size": 6,
        "xtick.major.width": 2,
        "ytick.major.width": 2,
        "xtick.minor.size": 2,
        "ytick.minor.size": 2,
    }
)


In [ ]:
def draw_carpet(ax, x, y_bottom, y_top, x_top=2, x_bottom=8,
                x_shift=0, y_shift=0, r=0.1, p=0.8, cmap='crest_r'):

    xs = x + x_shift          # <-- horizontal shift
    yb = y_bottom + y_shift
    yt = y_top + y_shift

    i_top = np.searchsorted(x, x_top)
    i_bottom = np.searchsorted(x, x_bottom)

    poly_x = np.concatenate([xs[:i_bottom],
                             [xs[i_bottom-1], xs[-1]],
                             xs[i_top:][::-1],
                             [xs[i_top], xs[0]]])

    poly_y = np.concatenate([yb[:i_bottom],
                             [yb[i_bottom-1], yt[-1]],
                             yt[i_top:][::-1],
                             [yt[i_top], yb[0]]])

    X, Y = np.meshgrid(np.linspace(-2,12,400),
                       np.linspace(-8,6,400))

    x0 = (x_top + x_bottom) / 2 + x_shift
    y0 = (np.mean(yt[i_top:i_bottom]) + np.mean(yb[i_top:i_bottom])) / 2

    Z = ((X - x0)**2 + (Y - y0)**2)**(p/2) / (r**p)

    path = Path(np.column_stack([poly_x, poly_y]))
    mask = ~path.contains_points(np.column_stack([X.ravel(), Y.ravel()]))
    Z = np.ma.array(Z, mask=mask.reshape(X.shape))

    ax.contourf(X, Y, Z, levels=21, cmap=cmap, alpha=1)
    ax.contour(X, Y, Z, levels=21, colors='black', linewidths=0.7, alpha=0.7)

    ax.plot(xs[:i_bottom], yb[:i_bottom], 'k', linewidth=2)
    ax.plot(xs[i_top:], yt[i_top:], 'k', linewidth=2)
    ax.plot([xs[0], xs[i_top]], [yb[0], yt[i_top]], 'k', linewidth=2)
    ax.plot([xs[i_bottom-1], xs[-1]],
            [yb[i_bottom-1], yt[-1]], 'k', linewidth=2)



# ---- your curves ----
x = np.linspace(0, 10, 400)
y_bottom = 0.4*np.sin(0.5*x)
y_top    = 4 + 0.4*np.sin(0.5*x)

fig, ax = plt.subplots(figsize=(8,5))


# Custom cmap based on #403D79 (light → base color)
custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom_403D79",
    [(0.0, "#FFFFFF"),      # pure white
     (0.3, "#FFFFFF"),      # keep white longer (increase 0.4 → 0.5 for even more)
     (1.0, '#0085C7')]
)


# top carpet
draw_carpet(ax, x, y_bottom, y_top,
            x_shift=0, y_shift=0,
            r=1, p=0.8, cmap=custom_cmap)



# Custom cmap based on #403D79 (light → base color)
custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom_40B7AD",
    [(0.0, "#FFFFFF"),      # pure white
     (0.3, "#FFFFFF"),      # keep white longer (increase 0.4 → 0.5 for even more)
     (1.0, "#DF0024")]
)



# bottom carpet shifted left
draw_carpet(ax, x, y_bottom, y_top,
            x_shift=-0.2, y_shift=-6,
            r=1, p=0.8, cmap=custom_cmap)


#3###################################
# ---------- trajectory ON TOP (red) carpet ----------
# recompute the red-carpet "center" (same as inside draw_carpet for y_shift=0)
x0 = (2 + 8) / 2
i_top = np.searchsorted(x, 2)
i_bottom = np.searchsorted(x, 8)
y0 = (np.mean(y_top[i_top:i_bottom]) + np.mean(y_bottom[i_top:i_bottom])) / 2

# helpers: top/bottom curves at any x (linear interp)
def yb(xq): return np.interp(xq, x, y_bottom)
def yt(xq): return np.interp(xq, x, y_top)

# start point (inside, far from center => high loss)
x_start = 1.7
t = np.linspace(0, 1, 35)
x_traj = x_start + (x0 - x_start) * t

# keep trajectory inside: follow the midline + small offset that vanishes at the minimum
mid = 0.5 * (yb(x_traj) + yt(x_traj))
offset = 0.2 * (1 - t)                      # starts higher, goes to 0 at the min
y_traj = mid + offset

# clamp strictly inside the carpet
eps = 0.01
y_traj = np.clip(y_traj, yb(x_traj) + eps, yt(x_traj) - eps)



# draw trajectory + arrow head
ax.plot(x_traj[:-5], y_traj[:-5], color=sns.color_palette('tab10')[0], linewidth=2, zorder=20)
ax.annotate('', xy=(x_traj[-1], y_traj[-1]), xytext=(x_traj[-3], y_traj[-3]),
            arrowprops=dict(arrowstyle='-|>', lw=4, color=sns.color_palette('tab10')[0]))

ax.plot( x_traj[0], y_traj[0], "x", ms=8, color='k', markeredgecolor='k', markeredgewidth=1.5, zorder=50, alpha=1, )

# ---------- translated + curly trajectory on BOTTOM carpet ----------
x_shift_bot = -0.2
y_shift_bot = -6

# bottom curves (same functions, just shifted in y and x)
def yb_bot(xq): return np.interp(xq - x_shift_bot, x, y_bottom) + y_shift_bot
def yt_bot(xq): return np.interp(xq - x_shift_bot, x, y_top)    + y_shift_bot

# translate the top trajectory endpoints into bottom carpet coordinates
x2 = x_traj + x_shift_bot
t2 = np.linspace(0, 1, len(x2))

# follow bottom midline, add a decaying wiggle so it ends at the minimum
mid2 = 0.5 * (yb_bot(x2) + yt_bot(x2))
amp = 1.5 # * (1 - t2)                 # bigger at start, 0 at the end
wiggle = amp * np.sin(1*np.pi*t2)    # "curly" (increase 12 for more curls)
y2 = mid2 + wiggle

# keep it strictly inside the bottom carpet
eps2 = 0.02
y2 = np.clip(y2, yb_bot(x2) + eps2, yt_bot(x2) - eps2)

# draw bottom trajectory + arrow + markers
ax.plot(x2[:-1], y2[:-1], color=sns.color_palette('tab10')[3], linewidth=2, zorder=20)
ax.annotate('', xy=(x2[-1], y2[-1]), xytext=(x2[-3], y2[-3]),
            arrowprops=dict(arrowstyle='-|>', lw=4, color=sns.color_palette('tab10')[3]))

ax.plot(x2[0], y2[0], "x", ms=8, color='k', markeredgewidth=1.5, zorder=50)


# --- dashed line (no arrow head) ---
ax.plot([x2[0], x2[-1]-0.2],
        [y2[0], y2[-1]],
        linestyle='--',
        color='k',
        lw=1.5,
        zorder=49)

# --- solid arrow head only (small final segment) ---
tip_frac = 0.93   # controls how short the solid tip segment is

x_tip = x2[0] + tip_frac * (x2[-1] - x2[0])
y_tip = y2[0] + tip_frac * (y2[-1] - y2[0])

arrow = FancyArrowPatch(
    posA=(x_tip, y_tip),
    posB=(x2[-1], y2[-1]),
    arrowstyle='-|>',
    mutation_scale=20,
    lw=1.5,
    color='k',
    zorder=50
)
ax.add_patch(arrow)




# --- connect with index offset ---
offset = 1

# avoid first and last
idx = np.linspace(6, len(x_traj)-2-offset, 5).astype(int)

for i in idx:
    j = i + offset
    if 0 <= j < len(x2)-1:
        # ax.plot([x_traj[i], x2[j]],
        #         [y_traj[i], y2[j]],
        #         color='k', linestyle='--', linewidth=1, alpha=0.8, zorder=10)

        ax.annotate('',
                xy=(x2[j], y2[j]),              # arrow tip (bottom)
                xytext=(x_traj[i], y_traj[i]),  # start (top)
                arrowprops=dict(arrowstyle='-|>',
                                lw=1.,
                                linestyle='--',
                                color='k',
                                alpha=1,
                                shrinkA=0,
                                mutation_scale=10, 
                                shrinkB=0), zorder=2)


ax.annotate('',
        xy=(x2[0], y2[0]),              # arrow tip (bottom)
        xytext=(x_traj[0], y_traj[0]),  # start (top)
        arrowprops=dict(arrowstyle='-|>',
                        lw=1.,
                        linestyle='--',
                        color='k',
                        alpha=1,
                        shrinkA=0,
                        mutation_scale=10, 
                        shrinkB=0), zorder=2)




ax.text(1,3.4,r'$\theta$',size=28)
ax.text(0.3,-3.5,r'$\sigma$',size=28)

ax.text(3.5,4.75,r'$\text{parameter space}$',size=20,rotation=0)
ax.text(2.85,-6.9,r'$\text{overlap space}$',size=20,rotation=0)


# --- big green curved arrow (top-right -> bottom-right) ---
from matplotlib.patches import FancyArrowPatch
arrow = FancyArrowPatch(
    posA=(9.2, 1.8),        # start (top carpet right side)
    posB=(8.7, -2.2),       # end   (bottom carpet right side)
    connectionstyle="arc3,rad=-0.35",  # curvature (tune rad)
    arrowstyle='-|>',
    mutation_scale=30,
    lw=2,
    color='k',
    zorder=50
)
ax.add_patch(arrow)

# ax.text(2.5,-1.0,r'$G(\theta)$',color='k',size=22,zorder=50)

ax.text(2.5, -0.9, r'$G(\theta)$',
        color='k',
        size=22,
        zorder=50,
        bbox=dict(facecolor='white', alpha=0.8))

# ax.set_title(r'$\text{Optimzation}$')

ax.axis('off')
# plt.savefig('../figures/1b.pdf', bbox_inches='tight')
plt.show()


In [ ]:
N = 500
dt = 0.05
t_max = 20
time_window = int(t_max/dt)
target_exp = [0.2]
lr = 5e-3
num_epochs = 2_000

# # ---------- TRAIN ----------
# m,u,v,z = initialize_vectors(N, 1)
# m = normalize(m) * np.sqrt(N)
# u = normalize(u) * np.sqrt(N)
# v = normalize(v) * np.sqrt(N)
# z = normalize(z) * np.sqrt(N)
# # m,u,v,z = initialize_vectors_balanced(N, 1)

# loss, params, _ = train_full_model( target_exp, N, num_epochs, time_window=time_window, m0=m, z0=z, u0=u, v0=v, lr=N*lr, dt=dt)
# all_full_rnn = vectors_to_overlaps_rank_1(N, params, num_epochs)
# all_loss_eff, all_params_eff, _ = run_10d_ode(params[0], N, dt, target_exp[0], num_epochs, time_window, lr, None, False )
# all_loss_eff_4d, all_params_eff_4d, _ = run_10d_ode(params[0], N, dt, target_exp[0], num_epochs, time_window, lr, None, True )

# # ---------- SAVE ----------
# data_to_save = {
#     "loss": loss,
#     "all_loss_eff": all_loss_eff,
#     "all_loss_eff_4d": all_loss_eff_4d,
#     "all_full_rnn": all_full_rnn,
#     "all_params_eff": all_params_eff,
# }

# with open("../data/fig_1_and_5.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")

# ---------- LOAD ----------
with open("../data/fig_1_and_5.pkl", "rb") as f:
    loaded_data = pickle.load(f)

loss = loaded_data["loss"]
all_loss_eff = loaded_data["all_loss_eff"]
all_loss_eff_4d = loaded_data["all_loss_eff_4d"]
all_full_rnn = loaded_data["all_full_rnn"]
all_params_eff = loaded_data["all_params_eff"]
print("Loaded successfully fig_1_and_5")

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5.8,4))
ax.plot((np.array(loss))[:],lw=3,color="#0085C7",label=r'$\theta$'r' $\text{parameter-space}$')#r'$\nabla_{\theta}\mathcal{L}$')
ax.plot((np.array(all_loss_eff))[:],lw=2,color="#DF0024",ls='--',label=r'$\sigma$'r' $\text{overlap-space } (G(\theta))$')#r"${G}(\theta)\nabla_{\sigma}\mathcal{L}$")
ax.plot((np.array(all_loss_eff_4d))[:],lw=2,color='#000000',ls='--',label=r'$\sigma$'r' $\text{overlap-space (directly) }$')#r'$\nabla_{\sigma}\mathcal{L}$')

ax.set_xscale("log")
ax.set_xticks([10, 1000])
ax.set_yticks([0, 2])
ax.xaxis.set_minor_locator(NullLocator())
ax.legend(loc='lower left',frameon=False,fontsize=17)

ax.set_xlabel(r'$\text{Epoch}$',size=20,labelpad=-15)
ax.set_ylabel(r'$\text{Loss}$',size=20,labelpad=-5)
ax.set_title(r'$\text{Loss curve}$',size=22)

sns.despine()
plt.tight_layout()
# plt.savefig('../figures/1c.pdf', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(2, 3, figsize=(16, 8))
ax = ax.flatten()

# --- Panel 0: filter task ---
x = 0.5 * torch.randn((time_window, 1))
y_target = exponential_filter(x, target_exp[0], dt)
t_ = np.arange(time_window)*dt
ax[0].plot(t_,x.detach().flatten(), lw=1.5, alpha=0.4, color='k',
           label=r'$\text{Input}\,\, x$')
ax[0].plot(t_,y_target.detach().flatten(), lw=3,
           label=r'$\text{Target}\,\, y^{\star}$')

ax[0].set_yticks([-1, 0, 1])
ax[0].set_xticks([0, 20])
ax[0].set_title(r'$\text{Filter task}$', size=22)
ax[0].set_xlabel(r'$\text{Time (s)}$', size=18)
ax[0].set_ylabel(r'$\text{In/Out}$', size=18 ,labelpad=-5)
ax[0].legend(loc='lower center', frameon=False, fontsize=18, ncols=2)

# --- Panel 1: impulse response ---
sig_zm, sig_zu, sig_vm, sig_vu =  all_params_eff[-1][:4]
t_axis = torch.linspace(0.0, float(time_window), time_window)
x = torch.zeros((time_window, 1))
x[0, 0] = 1.0 / dt
y_target = torch.exp(-target_exp[0] * t_axis * dt).detach()
h = torch.zeros((2, 1))

y_pred = []
eff_rnn = effective_rnn_from_scaler(sig_zm, sig_zu, sig_vm, sig_vu, dt)  # identical, but don't recreate each step
for step in range(time_window):
    x_t = x[step, :].unsqueeze(0)
    y_hat, h = eff_rnn(x_t, h)
    y_pred.append(y_hat)

ax[1].plot(t_,y_target, lw=3, alpha=1, label=r'$\text{Target filter}$')
ax[1].plot(t_,torch.stack(y_pred).flatten().detach(), '--k',
           label=r'$\text{RNN filter}$')

ax[1].set_yticks([0,  1])
ax[1].set_xticks([0, 20])
ax[1].set_title(r'$\text{Impulse response}$', size=22)
ax[1].set_xlabel(r'$\text{Time (s)}$', size=18)
ax[1].set_ylabel(r'$\text{Output}$', size=18,labelpad=0)
ax[1].legend(loc='upper right', frameon=False, fontsize=18, ncols=1)

# --- Helper for the log-epoch panels (2,3,4) to keep them consistent ---
def style_log_epoch_axis(a, *, ylabel=None, ylim=None):
    a.set_xscale('log')
    a.xaxis.set_minor_locator(NullLocator())
    a.set_xticks([10, 1000])
    a.set_xlim(1, 2000)
    if ylabel is not None:
        a.set_ylabel(ylabel, size=18,labelpad=2)
    a.set_xlabel(r'$\text{Epoch}$', size=18)
    # if ylim is not None:
    #     a.set_ylim(ylim)

# --- Panel 2: loss curve ---
ax[2].plot(np.array(loss), lw=3, label=r'$\text{Simulation}$')
ax[2].plot(np.array(all_loss_eff), '--k', label=r'$\text{Theory}\,\,\text{(ODE)}$')

ax[2].set_title(r'$\text{Loss curve}$', size=22)
style_log_epoch_axis(ax[2], ylabel=r'$\text{Loss}$')
ax[2].set_yticks([0, 2])
ax[2].legend(loc='lower left', frameon=False, fontsize=18)

# --- Panel 3: loss-visible overlaps ---
lab = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$']
cols = sns.color_palette('Blues_r', n_colors=4)
for i in range(4):
    ax[3].plot(np.array(all_full_rnn)[:, i], lw=3, color=cols[i], label=lab[i])
    ax[3].plot(np.array(all_params_eff)[:, i], '--k')

ax[3].set_title(r'$\text{Loss-visible}$', size=22)
style_log_epoch_axis(ax[3], ylabel=r'$\text{Overlap}\,\,\, \boldsymbol{\sigma}$', ylim=(-0.2, 2.1))
ax[3].set_yticks([0, 2])
ax[3].legend(loc='upper left', frameon=False, fontsize=18, ncols=1)

# --- Panel 4: loss-invisible overlaps ---
lab = [r'$\sigma_{mu}$', r'$\sigma_{zv}$', r'$||m||^2$', r'$||u||^2$', r'$||v||^2$', r'$||z||^2$']
cols = sns.color_palette('Reds_r', n_colors=6)
for i in range(4, 10):
    ax[4].plot(np.array(all_full_rnn)[:, i], lw=3, color=cols[i - 4], label=lab[i - 4])
    ax[4].plot(np.array(all_params_eff)[:, i], '--k')

ax[4].set_title(r'$\text{Loss-invisible}$', size=22)
style_log_epoch_axis(ax[4], ylabel=r'$\text{Overlap}\,\,\, \tilde{\boldsymbol{\sigma}}$', ylim=(-0.2, 2.1))
ax[4].set_yticks([0, 2])
ax[4].legend(loc='center left', frameon=False, fontsize=18, ncols=1)

# --- Panel 5: final values ---
labels = [r'$\sigma_{zm}$', r'$\sigma_{zu}\cdot\sigma_{vm}$', r'$\sigma_{vu}$']
xpos = np.arange(len(labels)) + 0.5

ax[5].plot(xpos, [sig_zm, sig_zu * sig_vm, sig_vu], 'o', markersize=12,
           label=r'$\text{Simulation}$')
ax[5].plot(xpos, [1, 1 - target_exp[0], 1 - target_exp[0]], 'x', markersize=15,
           color='k', markeredgewidth=2, label=r'$\text{Theory}$')

ax[5].set_xticks(xpos)
ax[5].set_xticklabels(labels, rotation=0, ha='center', size=24)
ax[5].set_ylim([0.705, 1.05])
ax[5].set_yticks([0.8, 1])
ax[5].set_title(r'$\text{Final values}$', size=22)
ax[5].set_ylabel(r'$\text{Overlap}$', size=18,labelpad=-15)
ax[5].legend(loc='upper right', frameon=False, fontsize=15, ncols=1)

ax[5].text(x=0.46, y=1.03, s=r'$a^\star$', size=20)
ax[5].text(x=1.10, y=0.83, s=r'$a^\star(1-c^\star)$', size=20)
ax[5].text(x=2.35, y=0.83, s=r'$1-c^\star$', size=20)

# === Subplot labels ===
subplot_labels = [r'($\mathbf{a}$)', r'($\mathbf{b}$)', r'($\mathbf{c}$)', r'($\mathbf{d}$)', r'($\mathbf{e}$)',  r'($\mathbf{f\,}$)']
for i, label in enumerate(subplot_labels):
    ax[i].text(-0.07, 1.22, label, transform=ax[i].transAxes,
               fontsize=18, fontweight='bold', va='top', ha='right', font="Arial")

sns.despine()
plt.tight_layout()
plt.subplots_adjust(wspace=0.15)
plt.subplots_adjust(hspace=0.5)
# plt.savefig('../figures/fig_5.pdf', bbox_inches='tight')
plt.show()


In [ ]:
N = 500
dt = 0.05
t_max = 20
time_window = int(t_max/dt)
target_exp = [0.2]
lr = 5e-3
num_epochs = 2_000

# # ---------- TRAIN ----------
# _m,_u,_v,_z = initialize_vectors(N, 1)
# m0 = normalize(_m) * np.sqrt(N)
# u0 = normalize(_u) * np.sqrt(N)
# v0 = normalize(_v) * np.sqrt(N)
# z0 = normalize(_z) * np.sqrt(N)

# overlaps_theta_1 = (
#     (z0.T@m0).item()/N,
#     (v0.T@m0).item()/N,
#     (z0.T@u0).item()/N,
#     (v0.T@u0).item()/N,
    
#     (m0.T@u0).item()/N,
#     (z0.T@v0).item()/N,
#     (m0.T@m0).item()/N,
#     (u0.T@u0).item()/N,
#     (v0.T@v0).item()/N,
#     (z0.T@z0).item()/N,
# )

# m1, u1, v1, z1 = adjust_u_norm_and_mu_preserve_others(
#     m0, u0, v0, z0,
#     target_norm_u=32,
#     target_mu=400,
# )

# m1 = torch.Tensor(m1)
# u1 = torch.Tensor(u1)
# v1 = torch.Tensor(v1)
# z1 = torch.Tensor(z1)

# # check overlaps
# overlaps_theta_2 = (
#     (z1.T@m1).item()/N,
#     (v1.T@m1).item()/N,
#     (z1.T@u1).item()/N,
#     (v1.T@u1).item()/N,
    
#     (m1.T@u1).item()/N,
#     (z1.T@v1).item()/N,
#     (m1.T@m1).item()/N,
#     (u1.T@u1).item()/N,
#     (v1.T@v1).item()/N,
#     (z1.T@z1).item()/N,
# )

# print("Overlaps theta_1:", tuple(round(float(x), 3) for x in overlaps_theta_1))
# print("Overlaps theta_2:", tuple(round(float(x), 3) for x in overlaps_theta_2))

# loss_1, params_1, _ = train_full_model( target_exp, N, num_epochs, time_window=time_window, m0=m0, u0=u0, v0=v0, z0=z0, lr=N*lr, dt=dt)
# all_loss_eff_1, all_params_eff_1, _ = run_10d_ode(params_1[0], N, dt, target_exp[0], num_epochs, time_window, lr, None, False )

# loss_2, params_2, _ = train_full_model( target_exp, N, num_epochs, time_window=time_window, m0=m1, u0=u1, v0=v1, z0=z1, lr=N*lr, dt=dt)
# all_loss_eff_2, all_params_eff_2, _ = run_10d_ode(params_2[0], N, dt, target_exp[0], num_epochs, time_window, lr, None, False )

# # ---------- SAVE ----------
# data_to_save = {
#     "loss_1": loss_1,
#     "params_1": params_1,
#     "all_loss_eff_1": all_loss_eff_1,
#     "all_params_eff_1": all_params_eff_1,
#     "loss_2": loss_2,
#     "params_2": params_2,
#     "all_loss_eff_2": all_loss_eff_2,
#     "all_params_eff_2": all_params_eff_2,
# }

# with open("../data/fig_2.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")


# ---------- LOAD ----------
with open("../data/fig_2.pkl", "rb") as f:
    loaded_data = pickle.load(f)

loss_1 = loaded_data["loss_1"]
params_1 = loaded_data["params_1"]
all_loss_eff_1 = loaded_data["all_loss_eff_1"]
loss_2 = loaded_data["loss_2"]
params_2 = loaded_data["params_2"]
all_loss_eff_2 = loaded_data["all_loss_eff_2"]

print("Loaded successfully fig_2")

In [ ]:
def load_params(rnn, m, u, v, z):
    with torch.no_grad():
        rnn.m.data.copy_(m)
        rnn.u.data.copy_(u)
        rnn.v.data.copy_(v)
        rnn.z.data.copy_(z)

def rollout(rnn, x, noise=None):
    B, T = x.shape
    h = torch.zeros(rnn.N, B)
    y_out = []
    h_out = []

    for t in range(T):
        x_t = x[:, t].unsqueeze(0)
        y_t, h = rnn(x_t, h)

        if noise is not None:
            h = h + noise[t]

        h_out.append(h.clone())  # important: avoid in-place reference issues
        y_out.append(y_t.T.unsqueeze(1))

    y_out = torch.stack(y_out).flatten().detach().numpy()
    
    return y_out, h_out

# --- init ---
x = torch.randn((1, time_window))
B, T = x.shape

rnn_1 = low_rank_rnn(N=N, scale=1, rank=1, phi='linear', dt=dt, tau=1.0)
rnn_2 = low_rank_rnn(N=N, scale=1, rank=1, phi='linear', dt=dt, tau=1.0)

# --- params 1 (init) ---

overlaps_theta_1 = vectors_to_overlaps_rank_1(N,params_1,1)[0]
m, u, v, z = params_1[0]
load_params(rnn_1, m, u, v, z)
y1, h1 = rollout(rnn_1, x)

# --- params 2 (init) ---
overlaps_theta_2 = vectors_to_overlaps_rank_1(N,params_2,1)[0]
m, u, v, z = params_2[0]
load_params(rnn_2, m, u, v, z)
y2, h2 = rollout(rnn_2, x)

# --- noisy rollout ---
perturbation_noise = 0.5 * torch.randn((time_window, N, 1))

y11, h11 = rollout(rnn_1, x, noise=perturbation_noise)
y22, h22 = rollout(rnn_2, x, noise=perturbation_noise)

# --- params 1 (epoch 50) ---
m, u, v, z = params_1[300]
load_params(rnn_1, m, u, v, z)
y111, h111 = rollout(rnn_1, x)

# --- params 2 (epoch 50) ---
m, u, v, z = params_2[300]
load_params(rnn_2, m, u, v, z)
y222, h222 = rollout(rnn_2, x)

# ---------------------------
# Figure layout (side-by-side)
# ---------------------------
fig = plt.figure(figsize=(15, 4.4))

# overlaps | 2x2 block | loss
gs = fig.add_gridspec(
    nrows=2, ncols=4,
    width_ratios=[0.6, 1, 1, 2.],
    wspace=0.35, hspace=0.3
)

# ---------------------------
# Left: overlaps
# ---------------------------
axO = fig.add_subplot(gs[:, 0])

# ---------------------------
# Middle: 2x2 block
# ---------------------------
ax00 = fig.add_subplot(gs[0, 1])
ax10 = fig.add_subplot(gs[1, 1], sharex=ax00)

ax01 = fig.add_subplot(gs[0, 2])
ax11 = fig.add_subplot(gs[1, 2], sharex=ax01)

# ---------------------------
# Right: loss
# ---------------------------
axL = fig.add_subplot(gs[:, 3])


t_ = np.arange(time_window)*dt
# ===========================
# 2x2 time-series block
# ===========================
ax00.set_title(r'$\text{hidden activity}$', size=17)
ax00.plot(t_[:200],torch.stack(h1).squeeze().detach().numpy()[:200,10:14], lw=1.5, color='#0085C7')
ax00.plot(t_[:200],torch.stack(h2).squeeze().detach().numpy()[:200,10:14], lw=1, color="#DF0024", ls="--")
ax00.set_ylabel(r'$h_i$')
ax00.set_ylim(-.4,.4)

ax01.set_title(r'$\text{output}$', size=17)
ax01.plot(t_[:200],y1[:200], lw=1.5, color='#0085C7', label=r'$\theta_1$')
ax01.plot(t_[:200],y2[:200], lw=1, color="#DF0024", ls="--", label=r'$\theta_2$')
ax01.set_ylabel(r'$\hat{y}$')
ax01.legend(loc='upper left', frameon=False, fontsize=16)

ax10.set_title(r'$\text{noise-perturbation}$', size=17)
# ax10.plot(torch.stack(h11).squeeze().detach().numpy()[:100,12:17], lw=1.5, color='#0085C7')
# ax10.plot(torch.stack(h22).squeeze().detach().numpy()[:100,12:17], lw=1, color="#DF0024", ls="--")
ax10.plot(t_[:200],y11[:200], lw=1.5, color='#0085C7')
ax10.plot(t_[:200],y22[:200], lw=1, color="#DF0024", ls="--")
ax10.set_ylabel(r'$h_i$')
ax10.set_ylabel(r'$\hat{y}$')
ax10.set_xlabel(r'$\text{Time (s)}$', size=18, labelpad=-8)

ax11.set_title(r'$\text{learning-perturbation}$', size=17)
ax11.plot(t_[:200],y111[:200], lw=1.5, color='#0085C7')
ax11.plot(t_[:200],y222[:200], lw=1, color="#DF0024", ls="--")
ax11.set_ylabel(r'$\hat{y}$')
ax11.set_xlabel(r'$\text{Time (s)}$', size=18, labelpad=-8)

# ticks

# ax10.set_yticks([-1.8, 2.0])


ax11.tick_params(axis='x', labelsize=15)
ax10.tick_params(axis='x', labelsize=15)

# remove top-row x labels
ax00.tick_params(axis='x', which='both', bottom=True, labelbottom=False)
ax01.tick_params(axis='x', which='both', bottom=True, labelbottom=False)

for a in [ax00, ax01, ax10, ax11]:
    a.set_yticklabels([])


# ===========================
# Overlaps panel
# ===========================
before = np.array(overlaps_theta_1, dtype=float)
after  = np.array(overlaps_theta_2,  dtype=float)

lab = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$',
       r'$\sigma_{mu}$', r'$\sigma_{zv}$', r'$||m||^2$', r'$||u||^2$', 
       r'$||v||^2$', r'$||z||^2$']

n = len(before)
ypos = np.arange(n)[::-1]

y_sep = ypos[3] - 0.5
axO.axhline(y_sep, color='k', lw=2.2)

axO.scatter(before, ypos, color='#0085C7', s=55, zorder=3)
axO.scatter(after, ypos, color='#DF0024', marker='x', s=80, lw=1.5, zorder=4)

axO.set_yticks(ypos)
axO.set_yticklabels(lab, fontsize=17)
axO.set_xlim(-0.5, 2.7)
axO.set_xticks([0, 1, 2])
axO.tick_params(axis='x', labelsize=17)
axO.tick_params(axis='y', length=4)
axO.set_title(r"$\text{Initial overlaps}$", size=19)

axO.text(1.8, np.mean(ypos[:4])+0.3, 
         r'$\mathrm{Visible}\ \boldsymbol{\sigma}$',
         va='center', ha='left', fontsize=16, rotation=-90)

axO.text(1.8, np.mean(ypos[4:])+1.4, 
         r'$\mathrm{Invisible}\ \tilde{\boldsymbol{\sigma}}$',
         va='center', ha='left', fontsize=16, rotation=-90)


# ===========================
# Loss panel
# ===========================
axL.plot(loss_1, lw=3, color='#0085C7', label=r'$\theta \text{-space } (\theta_1)$')
axL.plot(loss_2, lw=3, color="#DF0024", label=r'$\theta \text{-space } (\theta_2)$')
axL.plot(all_loss_eff_1, color="k", lw=2, ls='--', label=r'$\text{Theory (10D)}$')
axL.plot(all_loss_eff_2, color="k", lw=2, ls='--')

axL.set_xscale("log")
axL.set_xticks([10, 1000])
axL.xaxis.set_minor_locator(NullLocator())
axL.set_yticks([0, 2])
axL.set_xlabel(r'$\text{Epoch}$', size=18, labelpad=-18)
axL.set_ylabel(r'$\text{Loss}$', size=18, labelpad=-5)
axL.set_title(r'$\text{Loss curve}$', size=19)
axL.legend(loc='lower left', frameon=False, fontsize=16)


# ===========================
# Subplot labels
# ===========================
axO.text(-0.35, 1.1, r'($\mathbf{b}$)', transform=axO.transAxes,
         fontsize=17, fontweight='bold', va='top', ha='right', fontname="Arial")

axes = [ax00, ax01, ax10, ax11]
subplot_labels = [
    r'($\mathbf{c}$)',
    r'($\mathbf{d}$)',
    r'($\mathbf{e}$)',
    r'($\mathbf{f\,}$)',
]

for ax, label in zip(axes, subplot_labels):
    ax.text(-0.07, 1.22, label,
            transform=ax.transAxes,
            fontsize=17,
            fontweight='bold',
            va='top',
            ha='right',
            fontname="Arial")

axL.text(-0.04, 1.1, r'($\mathbf{g}$)', transform=axL.transAxes,
         fontsize=17, fontweight='bold', va='top', ha='right', fontname="Arial")


ax00.set_yticks([-0.3, 0.3])
ax01.set_yticks([-0.0008, 0.0004])
ax10.set_yticks([-0.13, 0.03])
ax11.set_yticks([-0.05, 0.3])


# ---------------------------
# Clean style
# ---------------------------
sns.despine(fig=fig)
plt.tight_layout()
# plt.savefig('../figures/2b.pdf', bbox_inches='tight')
plt.show()

In [ ]:
N = 500
dt = 0.05
t_max = 20
time_window = int(t_max/dt)
target_exp = [0.2]
lr = 5e-3
num_epochs_1 = 3000
num_epochs_2 = 1000
num_epochs_3 = 4000

# # ---------- TRAIN ----------
# m,u,v,z = params_2[0]
# loss_31, params_31, _ = train_full_model( target_exp, N, num_epochs_1, time_window=time_window, noise=0.00, m0=m, u0=u, v0=v, z0=z, lr=N*lr, dt=dt)

# target_exp = [0.4, 0.2, 0.4]
# _m,_u,_v,_z = params_31[-1]
# loss_32, params_32, _ = train_full_model( target_exp, N, num_epochs_2, time_window=time_window, noise=0.00, m0=_m, u0=_u, v0=_v, z0=_z, lr=N*lr, dt=dt)

# target_exp = [0.2]
# _m,_u,_v,_z = params_32[-1]
# loss_33, params_33, _ = train_full_model( target_exp, N, num_epochs_3, time_window=time_window, noise=0.1 ,m0=_m, u0=_u, v0=_v, z0=_z, lr=2*N*lr, dt=dt) 

# # ---------- SAVE ----------
# data_to_save = {
#     "loss_31": loss_31,
#     "params_31": params_31,
#     "loss_32": loss_32,
#     "params_32": params_32,
#     "loss_33": loss_33,
#     "params_33": params_33,

# }

# with open("../data/fig_3_6.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")

# ---------- LOAD ----------
with open("../data/fig_3_6.pkl", "rb") as f:
    loaded_data = pickle.load(f)

loss_31 = loaded_data["loss_31"]
params_31 = loaded_data["params_31"]
loss_32 = loaded_data["loss_32"]
params_32 = loaded_data["params_32"]
loss_33 = loaded_data["loss_33"]
params_33 = loaded_data["params_33"]

print("Loaded successfully fig_3_6")

all_overlaps_31 =  vectors_to_overlaps_rank_1(N,params_31,num_epochs_1)
all_overlaps_32 =  vectors_to_overlaps_rank_1(N,params_32,int(num_epochs_2*3))
all_overlaps_33 =  vectors_to_overlaps_rank_1(N,params_33,num_epochs_3)
all_full_rnn_fig3 = np.concatenate((all_overlaps_31,all_overlaps_32,all_overlaps_33))


all_K = [] 

for epoch in range(num_epochs_1):
    m,u,v,z = params_31[epoch]
    K0 = compute_K(m, u, v, z)
    all_K.append(K0)

for epoch in range(num_epochs_2*3):
    m,u,v,z = params_32[epoch]
    K0 = compute_K(m, u, v, z)
    all_K.append(K0)

for epoch in range(num_epochs_3):
    m,u,v,z = params_33[epoch]
    K0 = compute_K(m, u, v, z)
    all_K.append(K0)

all_K = torch.stack(all_K).detach().numpy()

In [ ]:
# ----------------------- constants -----------------------
FIGSIZE = (16, 4.4)
WIDTH_RATIOS = [0.4, 1, 0.35]

START_IDX = 2000
REF_IDX_FIG3 = 9999

C_VISIBLE = "#0081C8"
C_INVISIBLE = "#EE334E"
C_REF = "k"

LABEL_FONTSIZE = 16
TICK_FONTSIZE_SMALL = 12
TICK_FONTSIZE = 17
TITLE_FONTSIZE = 20

YLABEL_X = -0.07
YLABEL_Y = 0.00

LEGEND_FONTSIZE = 15
LEGEND_HANDLE_TEXTPAD = -0.4
LEGEND_MARKER_MS = 14
LEGEND_EDGE_W = 0.5

# ----------------------- helpers -----------------------
def hide_spines(ax_, *, top=True, right=True, bottom=False):
    if top: ax_.spines["top"].set_visible(False)
    if right: ax_.spines["right"].set_visible(False)
    if bottom: ax_.spines["bottom"].set_visible(False)

def style_trace_axis(ax_, *, ylim, yticks, xlim=None):
    hide_spines(ax_, top=True, right=True, bottom=True)
    ax_.set_xticks([])
    ax_.set_ylim(*ylim)
    ax_.set_yticks(list(yticks))
    if xlim is not None:
        ax_.set_xlim(*xlim)

def draw_solution_manifold(ax0):
    # two vertical bars
    for x in (0.3, 0.7):
        ax0.plot([x, x], [0.2, 0.8], lw=3, color="k")

    # clean axes
    ax0.set_xticks([0, 0])
    ax0.set_yticks([0, 0])
    ax0.tick_params(axis="both", labelleft=False, labelbottom=False)
    ax0.set_xlim(0.1, 0.9)
    ax0.set_ylim(0.1, 0.9)

    # titles / labels
    ax0.set_title(r"$\text{Solution manifold}$", size=20)
    ax0.set_xlabel(r"$\text{Visible}\,\,\boldsymbol{\sigma}$", labelpad=7)
    ax0.set_ylabel(r"$\text{Invisible} \,\, \tilde{\boldsymbol{\sigma}}$", labelpad=7)

    # labels A/B
    ax0.text(0.27, 0.82, "A", size=20)
    ax0.text(0.67, 0.82, "B", size=20)

    # points
    ax0.scatter([0.3, 0.7], [0.6, 0.6], s=70, edgecolor="k", zorder=10, color=C_VISIBLE)
    ax0.scatter([0.3], [0.4], s=70, edgecolor="k", zorder=10, color=C_INVISIBLE)

    # numbers
    ax0.text(0.48, 0.58, "1", size=20, color=C_VISIBLE)
    ax0.text(0.475, 0.38, "2", size=20, color=C_INVISIBLE)

    # arrows
    ax0.annotate(
        "", xy=(0.7, 0.6), xytext=(0.3, 0.6),
        arrowprops=dict(
            arrowstyle="-|>", color=C_VISIBLE, lw=1.5, mutation_scale=20,
            connectionstyle="arc3,rad=-0.2"
        )
    )
    ax0.annotate(
        "", xy=(0.7, 0.6), xytext=(0.3, 0.6),
        arrowprops=dict(
            arrowstyle="<|-", color=C_VISIBLE, lw=1.5, mutation_scale=20,
            connectionstyle="arc3,rad=0.2"
        )
    )
    ax0.annotate(
        "", xy=(0.3, 0.4), xytext=(0.7, 0.6),
        arrowprops=dict(
            arrowstyle="-|>", color=C_INVISIBLE, lw=2, mutation_scale=20,
            connectionstyle="arc3,rad=-0.2"
        )
    )


def plot_trace_with_ref(ax_, y, *, color, lw=2, ref_y=None, ref_ls="--",
                        ref_color=C_REF, ref_alpha=0.7, ref_lw=1.5):
    ax_.plot(y, lw=lw, color=color)
    if ref_y is not None:
        ax_.axhline(
            y=ref_y, xmin=0.015, xmax=1,
            ls=ref_ls, color=ref_color, alpha=ref_alpha, lw=ref_lw
        )

def draw_task_bar(task_ax, ax_ref, *, epochs, labels, faces, tcols,
                  ymin=-0.2, ymax=0.3, fontsize=17, edge_lw=1.8, vline_lw=2):
    """Draw ABABA bar aligned to ax_ref x-limits."""
    task_ax.set_yticks([])
    task_ax.tick_params(axis="x", bottom=False, labelbottom=False)
    for s in task_ax.spines.values():
        s.set_visible(False)
    task_ax.set_clip_on(False)

    xL, xR = ax_ref.get_xlim()
    W = sum(epochs)
    lengths = [(xR - xL) * e / W for e in epochs]

    x = xL
    for L, lab, fc, tc in zip(lengths, labels, faces, tcols):
        task_ax.axvspan(
            x, x + L, ymin=ymin, ymax=ymax,
            facecolor=fc, edgecolor="k", lw=edge_lw,
            clip_on=False, zorder=22
        )
        task_ax.vlines(
            [x, x + L], ymin, ymax,
            transform=task_ax.get_xaxis_transform(),
            color="k", lw=vline_lw, clip_on=False, zorder=22
        )
        task_ax.text(
            x + L / 2 - 10, (ymin + ymax) / 2 - 0.05, lab,
            transform=task_ax.get_xaxis_transform(),
            ha="center", va="center",
            color=tc, fontsize=fontsize,
            clip_on=False, zorder=22
        )
        x += L

def set_multiline_ylabel(ax_, title, symbol, *, size=LABEL_FONTSIZE,
                         x=YLABEL_X, y=YLABEL_Y):
    ax_.set_ylabel(rf"$\text{{{title}}}$" + "\n" + rf"${symbol}$",
                   size=size, rotation=0)
    ax_.yaxis.set_label_coords(x, y)

def add_panel_label(ax_, label, *, x=-0.03, y=1.1, fontsize=20):
    ax_.text(
        x, y, label,
        transform=ax_.transAxes,
        fontsize=fontsize,
        fontweight="bold",
        va="top",
        ha="right",
        fontname="Arial",
    )

def make_dot_legend_handles(colors, *, ms=LEGEND_MARKER_MS,
                            edgecolor="black", edgewidth=LEGEND_EDGE_W):
    from matplotlib.lines import Line2D
    return [
        Line2D([0], [0], marker=".", ms=ms, color=c,
               markeredgecolor=edgecolor, markeredgewidth=edgewidth, lw=0)
        for c in colors
    ]

def add_dot_legend(ax_, labels, colors, *, loc="upper left"):
    handles = make_dot_legend_handles(colors)
    ax_.legend(
        handles, labels,
        loc=loc, frameon=False,
        fontsize=LEGEND_FONTSIZE,
        handletextpad=LEGEND_HANDLE_TEXTPAD
    )

# ----------------------- main figure -----------------------
fig, ax = plt.subplots(1, 3, figsize=FIGSIZE, width_ratios=WIDTH_RATIOS)

# left panel
draw_solution_manifold(ax[0])

# ----------------------- middle stacked panel -----------------------
outer_spec = ax[1].get_subplotspec()
fig.delaxes(ax[1])

gs_inner = gridspec.GridSpecFromSubplotSpec(
    4, 1,
    subplot_spec=outer_spec,
    height_ratios=[0.7, 1, 1, 1],
    hspace=0.3,
)

# visible
ax_vis = fig.add_subplot(gs_inner[1])
style_trace_axis(ax_vis, ylim=(0.5, 0.9), yticks=(0.5, 0.9), xlim=(-100, 1000*4 + 4000)) 
y_vis = np.array(all_full_rnn_fig3)[START_IDX:, 3]
ref_vis = np.array(all_full_rnn_fig3)[START_IDX, 3]
plot_trace_with_ref(ax_vis, y_vis, color=C_VISIBLE, lw=2, ref_y=ref_vis)
ax_vis.tick_params(axis="y", labelsize=TICK_FONTSIZE_SMALL)



# invisible
ax_inv = fig.add_subplot(gs_inner[2], sharex=ax_vis)
style_trace_axis(ax_inv, ylim=(2.1, 2.5), yticks=(2.1, 2.5))
y_inv = np.array(all_full_rnn_fig3)[START_IDX:, 7]
ref_inv = np.array(all_full_rnn_fig3)[START_IDX, 7]
plot_trace_with_ref(ax_inv, y_inv, color=C_INVISIBLE, lw=2, ref_y=ref_inv)
ax_inv.tick_params(axis="y", labelsize=TICK_FONTSIZE_SMALL)

ax_k = fig.add_subplot(gs_inner[3], sharex=ax_vis)
hide_spines(ax_k, top=True, right=True, bottom=False)

k_series = [
    (all_K[START_IDX:, 0, 496], all_K[START_IDX, 0, 496]),
    (all_K[START_IDX:, 1, 166], all_K[START_IDX, 1, 166]),
    (all_K[START_IDX:, 1, 214], all_K[START_IDX, 1, 214]), 
]

olympic_colors = [
    "#0081C8",  # Blue
    "#FCB131",  # Yellow
    "#000000",  # Black
    "#00A651",  # Green
    "#EE334E"   # Red
]
cc = [olympic_colors[1], olympic_colors[2], olympic_colors[3]]

for (y, y0), c in zip(k_series, cc):
    ax_k.plot(y, lw=2, color=c)
    ax_k.axhline(y=y0, xmin=0.015, xmax=1, ls="--", color="k", alpha=0.7, lw=1.5)

ax_k.set_ylim(0.7, 1.2)
ax_k.set_yticks([0.7, 1.2])
ax_k.tick_params(axis="y", labelsize=TICK_FONTSIZE_SMALL)

ax_k.tick_params(axis="x", labelsize=TICK_FONTSIZE)
ax_k.set_xlabel(r"$\text{Epoch}$", size=TITLE_FONTSIZE, labelpad=0)


# ----------------------- right stacked panel -----------------------
outer_spec_c = ax[2].get_subplotspec()
fig.delaxes(ax[2])

gs_c = gridspec.GridSpecFromSubplotSpec(
    2, 1, subplot_spec=outer_spec_c,
    height_ratios=[1, 1], hspace=0.1
)
ax_c_top = fig.add_subplot(gs_c[0])
ax_c_bot = fig.add_subplot(gs_c[1], sharex=ax_c_top)

# A2 vs A1 (top)
A1 = np.array(all_full_rnn_fig3)[3000, :4]
A2 = np.array(all_full_rnn_fig3)[3000 + 2000, :4]
I1 = np.array(all_full_rnn_fig3)[3000, 4:]
I2 = np.array(all_full_rnn_fig3)[3000 + 2000, 4:]

ax_c_top.scatter(A1, A2, color=C_VISIBLE, s=60, edgecolor="k", lw=0.5)
ax_c_top.scatter(I1, I2, color=C_INVISIBLE, s=60, edgecolor="k", lw=0.5)
ax_c_top.plot([0.2, 2.5], [0.2, 2.5], "--k", lw=0.5)

# A3 vs A1 (bottom)
A3 = np.array(all_full_rnn_fig3)[-1, :4]
I3 = np.array(all_full_rnn_fig3)[-1, 4:]

ax_c_bot.scatter(A1, A3, color=C_VISIBLE, s=60, edgecolor="k", lw=0.5)
ax_c_bot.scatter(I1, I3, color=C_INVISIBLE, s=60, edgecolor="k", lw=0.5)
ax_c_bot.plot([0.2, 2.5], [0.2, 2.5], "--k", lw=0.5)

# ticks / labels
ax_c_bot.tick_params(axis="both", labelsize=TICK_FONTSIZE)
ax_c_top.tick_params(axis="y", labelsize=TICK_FONTSIZE)
ax_c_top.tick_params(axis="x", labelleft=False, labelbottom=False)

ax_c_bot.set_xlabel(r"$\text{A1}$", size=TITLE_FONTSIZE, labelpad=-6)
ax_c_top.set_ylabel(r"$\text{A2}$", size=TITLE_FONTSIZE, labelpad=-1)
ax_c_bot.set_ylabel(r"$\text{A3}$", size=TITLE_FONTSIZE, labelpad=-1)
ax_c_top.set_title(r"$\text{Final overlaps}$", size=TITLE_FONTSIZE)

add_dot_legend(
    ax_c_top,
    [r"$\text{Visible}$", r"$\text{Invisible}$"],
    [C_VISIBLE, C_INVISIBLE],
)


# ylabels for stacked traces
set_multiline_ylabel(ax_vis, "Visible", r"\sigma")
set_multiline_ylabel(ax_inv, "Invisible", r"\tilde{\sigma}")
set_multiline_ylabel(ax_k, "Invariant", r"K_{ij}")


ax_c_bot.annotate(
    "", xy=(2.4, 2), xytext=(2.40, 1.55),
     arrowprops=dict(arrowstyle="-|>", color='k', lw=1.4, mutation_scale=16,))


ax_c_bot.annotate(
    "", xy=(.8, .7), xytext=(.8,0.25),
     arrowprops=dict(arrowstyle="-|>", color='k', lw=1.4, mutation_scale=16,))


ax_c_top.annotate(
    "", xy=(2.4, 2.3), xytext=(2.40, 1.85),
     arrowprops=dict(arrowstyle="-|>", color='k', lw=1.4, mutation_scale=16,))


ax_c_top.annotate(
    "", xy=(.8, .7), xytext=(.8,0.25),
     arrowprops=dict(arrowstyle="-|>", color='k', lw=1.4, mutation_scale=16,))

sns.despine(ax=ax[0])
sns.despine(ax=ax_c_top)
sns.despine(ax=ax_c_bot)


ax_vis.text(800, 0.42, r"$\text{A1}$", size=18)
ax_vis.text(2750, 0.42, r"$\text{A2}$", size=18)
ax_vis.text(7750, 0.42, r"$\text{A3}$", size=18)

# task bar
task_ax = fig.add_subplot(gs_inner[0], sharex=ax_vis)
draw_task_bar(
    task_ax, ax_vis,
    epochs=[1090, 1000, 1015, 980, 4000],
    labels=["A", "B", "A", "B", "A (+noise)"],
    faces=["white", "black", "white", "black", "white"],
    tcols=["k", "w", "k", "w", "k"],
    ymin=0.0, ymax=0.7
)
task_ax.set_title(r"$\text{A-B-A protocol}$", size=TITLE_FONTSIZE)

# vertical markers on task bar
for x0 in [980, 2000, 3000, 4000, 8000]: ###########################################################
    task_ax.vlines(
        x0, -5.5, 0.1,
        transform=task_ax.get_xaxis_transform(),
        color="k", lw=1.0, alpha=0.2,
        clip_on=False, zorder=40
    )

# layering
task_ax.set_zorder(10)
for a in (ax_vis, ax_inv, ax_k):
    a.set_zorder(1)
task_ax.patch.set_alpha(0)

# shared x ticks off
for a in (ax_vis, ax_inv):
    a.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

ax_k.set_xticks([1000, 3000, 4000, 8000 ])
ax_k.set_xticklabels([1000, 3000, 4000, 8000  ])

# panel labels
add_panel_label(ax[0], r"($\mathbf{a}$)")
task_ax.text(-0.05, 1.65, r"($\mathbf{b}$)", transform=task_ax.transAxes,
             fontsize=18, fontweight="bold", va="top", ha="right", font="Arial")

ax_c_top.text(-0.05, 1.22, r"($\mathbf{c}$)", transform=ax_c_top.transAxes,
              fontsize=18, fontweight="bold", va="top", ha="right", font="Arial")


plt.tight_layout()
plt.subplots_adjust(wspace=0.22)
# plt.savefig("../figures/fig_3.pdf", bbox_inches='tight')
plt.show()

In [ ]:
data = np.array(all_full_rnn_fig3)

lab = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$',
       r'$\sigma_{mu}$', r'$\sigma_{zv}$', r'$||m||^2$', r'$||u||^2$', 
       r'$||v||^2$', r'$||z||^2$']

fig, axes = plt.subplots(5, 2, figsize=(11, 7), sharex=True)

for i, ax in enumerate(axes.flat):
    color = "#0085C7" if i < 4 else "#DF0024"  
    
    # Use x_range to align with specific epochs
    ax.plot(data[2000:, i],color=color)
    ax.axhline(y=data[3000, i],ls='--',color='gray')
    ax.tick_params(labelsize=14)
    
    # Calculate custom y-ticks: 5% padding from the limits
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    ticks = [y_min + 0.05 * y_range, y_max - 0.05 * y_range]
    
    # Set the ticks and the 2nd decimal precision formatter
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    ax.set_xticks([2000, 6000])

    ax.set_ylabel(lab[i], size=20, rotation=0, labelpad=0, va='center')
    ax.tick_params(axis='y', labelsize=12)
    ax.tick_params(axis='x', labelsize=14)

# Labels and annotations
for ax in axes[4, :]:
    ax.set_xlabel(r'Epoch', size=18)

axes[4,0].set_xlabel(r'$\text{Epoch}$',size=18)
axes[4,1].set_xlabel(r'$\text{Epoch}$',size=18)

axes[0,0].text(300,1.05,s='A')
axes[0,0].text(1300,1.05,s='B')
axes[0,0].text(2300,1.05,s='A')
axes[0,0].text(3300,1.05,s='B')
axes[0,0].text(5000,1.05,s='A+noise')

axes[0,1].text(300,0.97,s='A')
axes[0,1].text(1300,0.97,s='B')
axes[0,1].text(2300,0.97,s='A')
axes[0,1].text(3300,0.97,s='B')
axes[0,1].text(5000,0.97,s='A+noise')

from matplotlib.patches import ConnectionPatch

# Define the epochs for the vertical lines
epochs = [1000, 2000, 3000, 4000]

for col in [0, 1]:
    for x in epochs:
        # Create a connection from the top of the top plot to the bottom of the bottom plot
        con = ConnectionPatch(
            xyA=(x, 1.2),  # Point A: Above the top row (in axis coords)
            xyB=(x, -0.00), # Point B: Below the bottom row (in axis coords)
            coordsA=axes[0, col].get_xaxis_transform(),
            coordsB=axes[4, col].get_xaxis_transform(),
            axesA=axes[0, col],
            axesB=axes[4, col],
            color="k",
            linestyle="-",
            linewidth=1,
            alpha=0.2,
            zorder=100  # Ensure it is on top of everything
        )
        fig.add_artist(con)


plt.tight_layout()
sns.despine()
# plt.savefig("../figures/fig_6.pdf", bbox_inches='tight')
plt.show()

In [ ]:
N = 500
dt = 0.05
t_max = 20
time_window = int(t_max/dt)
target_exp = [0.2,0.4,0.2,0.4,0.2]
lr = 1e-3
num_epochs = 4000

# # ---------- TRAIN ----------
# m,u,v,z = params_2[0]
# loss_adam, params_adam, _ = train_full_model_adam( target_exp, N, num_epochs, time_window=time_window, noise=0.00, m0=m,  u0=u, v0=v, z0=z,lr=lr, dt=dt)
# all_full_rnn_adam = vectors_to_overlaps_rank_1(N, params_adam, int(5*num_epochs))

# # # ---------- SAVE ----------
# data_to_save = {
#     "loss_adam": loss_adam,
#     "all_full_rnn_adam": all_full_rnn_adam,
# }

# with open("../data/fig_7.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")

# ---------- LOAD ----------
with open("../data/fig_7.pkl", "rb") as f:
    loaded_data = pickle.load(f)

loss_adam = loaded_data["loss_adam"]
all_full_rnn_adam = loaded_data["all_full_rnn_adam"]

print("Loaded successfully fig_7")

In [ ]:
data = np.array(all_full_rnn_adam)

lab = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$',
       r'$\sigma_{mu}$', r'$\sigma_{zv}$', r'$||m||^2$', r'$||u||^2$', 
       r'$||v||^2$', r'$||z||^2$']

fig, axes = plt.subplots(5, 2, figsize=(11, 7), sharex=True)

for i, ax in enumerate(axes.flat):
    color = "#0085C7" if i < 4 else "#DF0024"  
    
    # Use x_range to align with specific epochs
    ax.plot(data[:, i],color=color)
    ax.axhline(y=data[3000, i],ls='--',color='gray')
    ax.tick_params(labelsize=14)
    
    # Calculate custom y-ticks: 5% padding from the limits
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    ticks = [y_min + 0.05 * y_range, y_max - 0.05 * y_range]
    
    # Set the ticks and the 2nd decimal precision formatter
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    ax.set_ylabel(lab[i], size=20, rotation=0, labelpad=0, va='center')
    ax.tick_params(axis='y', labelsize=12)
    ax.tick_params(axis='x', labelsize=14)

    ax.set_xticks([5000, 15000])

# Labels and annotations
for ax in axes[4, :]:
    ax.set_xlabel(r'$\text{Epoch}$', size=18)

axes[0,0].text(2000,1.1,s='A')
axes[0,0].text(5500,1.1,s='B')
axes[0,0].text(9500,1.1,s='A')
axes[0,0].text(13500,1.1,s='B')
axes[0,0].text(17500,1.1,s='A')

axes[0,1].text(2000,1.09,s='A')
axes[0,1].text(5500,1.09,s='B')
axes[0,1].text(9500,1.09,s='A')
axes[0,1].text(13500,1.09,s='B')
axes[0,1].text(17500,1.09,s='A')

# Define the epochs for the vertical lines
epochs = [4000, 8000, 12000, 16000]

for col in [0, 1]:
    for x in epochs:
        # Create a connection from the top of the top plot to the bottom of the bottom plot
        con = ConnectionPatch(
            xyA=(x, 1.2),  # Point A: Above the top row (in axis coords)
            xyB=(x, -0.00), # Point B: Below the bottom row (in axis coords)
            coordsA=axes[0, col].get_xaxis_transform(),
            coordsB=axes[4, col].get_xaxis_transform(),
            axesA=axes[0, col],
            axesB=axes[4, col],
            color="k",
            linestyle="-",
            linewidth=1,
            alpha=0.2,
            zorder=100  # Ensure it is on top of everything
        )
        fig.add_artist(con)

plt.tight_layout()
sns.despine()
# plt.savefig("../figures/fig_7.pdf", bbox_inches='tight')
plt.show()

In [ ]:
N = 600
dt = 0.025
t_max = 20
batch_size = 10
num_epochs = 2_000
lr = 5e-2
rnn = low_rank_rnn(N=N, scale=1, rank=1,  phi='erf', dt=dt, tau=1)

# # ---------- TRAIN ----------
# m,u,v,z = initialize_vectors_balanced(N, 1)
# with torch.no_grad():     
#     rnn.m.data.copy_(m)
#     rnn.u.data.copy_(u)
#     rnn.v.data.copy_(v)
#     rnn.z.data.copy_(z)
    
# optimizer = optim.SGD([p for p in rnn.parameters() if p.requires_grad], lr=N*lr)
# losses = []
# params = []
# for step in range(num_epochs+1):
    
#     params.append(deepcopy(rnn))
    
#     x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(4.0,10.0), input_amp=1.0, target_amp=0.5, use_torch=True) 
#     B, T, _ = x.shape
#     h = torch.zeros(rnn.N, B)     # (N,B)
    
#     ys = []
#     for t in range(T):
#         x_t = x[:, t, :].T                 # (1,B)
#         y_t, h = rnn(x_t, h)               # y_t: (1,B)
#         ys.append(y_t.T.unsqueeze(1))      # -> (B,1,1)
        
#     yhat = torch.cat(ys, dim=1)            # (B,T,1)
#     _loss = masked_mse(yhat, y, mask)
    
#     losses.append(_loss.item())
    
#     optimizer.zero_grad()
#     _loss.backward()
#     optimizer.step()

#     if step % 500 == 0 or step == 1:
#         with torch.no_grad():
#             _mse = _loss.item()
#         print(f"step {step:5d} | masked mse {_mse:.4f}")

# rnn = params[0]
# m,u,v,z = rnn.m.detach().numpy().flatten(), rnn.u.detach().numpy().flatten(), rnn.v.detach().numpy().flatten(), rnn.z.detach().numpy().flatten()
# sig_zm = z@m/N
# sig_zu = z@u/N
# sig_vm = v@m/N
# sig_vu = v@u/N

# sig_mu = m@u/N
# sig_zv = z@v/N

# mm  = m@m/N
# uu  = u@u/N
# vv  = v@v/N
# zz  = z@z/N

# all_10d_rnn_nl = [] 
# all_loss_10d_nl = [] 

# for epoch in range(num_epochs):
#     all_10d_rnn_nl.append([sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz])

#     eff_rnn_nl = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
#     optimizer  = torch.optim.SGD(eff_rnn_nl.parameters(), lr=lr) 
    
#     x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(4.0,10.0), input_amp=1.0, target_amp=0.5, use_torch=True) 
#     B, T, _ = x.shape
#     h = torch.zeros(2, B)     
    
#     ys = []
#     for t in range(T):
#         x_t = x[:, t, :].T                 
#         y_t, h = eff_rnn_nl(x_t, h)        
#         ys.append(y_t.T.unsqueeze(1))      
        
#     yhat = torch.cat(ys, dim=1)            
#     _loss = masked_mse(yhat, y, mask)
    
#     all_loss_10d_nl.append(_loss.item())
#     optimizer.zero_grad()
#     _loss.backward()

#     d_zm, d_zu = eff_rnn_nl.zm.grad.item(),eff_rnn_nl.zu.grad.item() 
#     d_vm, d_vu = eff_rnn_nl.vm.grad.item(),eff_rnn_nl.vu.grad.item()
#     d_mu = eff_rnn_nl.mu.grad.item()
#     d_mm, d_uu = eff_rnn_nl.mm.grad.item(),eff_rnn_nl.uu.grad.item()
    
#     scale = 1.0     
#     #############################################
#     sig_zm -= scale * lr * ( (mm + zz) * d_zm + sig_mu * d_zu + sig_zv * d_vm + sig_zu * d_mu + 2.0 * sig_zm * d_mm )
#     sig_zu -= scale * lr * ( sig_mu * d_zm + (uu + zz) * d_zu + sig_zv * d_vu + sig_zm * d_mu + 2.0 * sig_zu * d_uu )
#     sig_vm -= scale * lr * ( (mm + vv) * d_vm + sig_mu * d_vu + sig_zv * d_zm + sig_vu * d_mu + 2.0 * sig_vm * d_mm )
#     sig_vu -= scale * lr * ( sig_mu * d_vm + (uu + vv) * d_vu + sig_zv * d_zu + sig_vm * d_mu + 2.0 * sig_vu * d_uu )
    
#     sig_mu -= scale * lr * ( sig_zu * d_zm + sig_zm * d_zu + sig_vu * d_vm + sig_vm * d_vu + (mm + uu) * d_mu + 2.0 * sig_mu * (d_mm + d_uu) )
#     sig_zv -= scale * lr * ( sig_vm * d_zm + sig_vu * d_zu + sig_zm * d_vm + sig_zu * d_vu )

#     mm -= scale * lr * ( 2.0 * (sig_zm * d_zm + sig_vm * d_vm + sig_mu * d_mu + 2.0 * mm * d_mm ) )
#     uu -= scale * lr * ( 2.0 * (sig_zu * d_zu + sig_vu * d_vu + sig_mu * d_mu + 2.0 * uu * d_uu ) )
#     vv -= scale * lr * ( 2.0 * (sig_vm * d_vm + sig_vu * d_vu) ) 
#     zz -= scale * lr * ( 2.0 * (sig_zm * d_zm + sig_zu * d_zu) )

# rnn = params[0]
# m,u,v,z = rnn.m.detach().numpy().flatten(), rnn.u.detach().numpy().flatten(), rnn.v.detach().numpy().flatten(), rnn.z.detach().numpy().flatten()
# sig_zm = z@m/N
# sig_zu = z@u/N
# sig_vm = v@m/N
# sig_vu = v@u/N

# sig_mu = m@u/N
# sig_zv = z@v/N

# mm  = m@m/N
# uu  = u@u/N
# vv  = v@v/N
# zz  = z@z/N

# all_10d_rnn_nl_w = [] 
# all_loss_10d_nl_w = [] 

# for epoch in range(num_epochs):
#     all_10d_rnn_nl_w.append([sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz])

#     eff_rnn_nl = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
#     optimizer  = torch.optim.SGD(eff_rnn_nl.parameters(), lr=lr) 
    
#     x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(4.0,10.0), input_amp=1.0, target_amp=0.5, use_torch=True) 
#     B, T, _ = x.shape
#     h = torch.zeros(2, B)     
    
#     ys = []
#     for t in range(T):
#         x_t = x[:, t, :].T                 
#         y_t, h = eff_rnn_nl(x_t, h)        
#         ys.append(y_t.T.unsqueeze(1))      
        
#     yhat = torch.cat(ys, dim=1)            
#     _loss = masked_mse(yhat, y, mask)
    
#     all_loss_10d_nl_w.append(_loss.item())
#     optimizer.zero_grad()
#     _loss.backward()

#     d_zm, d_zu = eff_rnn_nl.zm.grad.item(),eff_rnn_nl.zu.grad.item() 
#     d_vm, d_vu = eff_rnn_nl.vm.grad.item(),eff_rnn_nl.vu.grad.item()
#     d_mu = eff_rnn_nl.mu.grad.item()
#     d_mm, d_uu = eff_rnn_nl.mm.grad.item(),eff_rnn_nl.uu.grad.item()
    
#     scale = 1.0 
#     #############################################
#     sig_zm -= scale * lr * d_zm
#     sig_vm -= scale * lr * d_vm
#     sig_zu -= scale * lr * d_zu
#     sig_vu -= scale * lr * d_vu
    
#     sig_mu -= scale * lr * d_mu
#     mm -= scale * lr * d_mm
#     uu -= scale * lr * d_uu


# params_np = []
# for ep in range(2000):
#     rnn = params[ep]
#     m = rnn.m.detach().cpu().numpy().flatten()
#     u = rnn.u.detach().cpu().numpy().flatten()
#     v = rnn.v.detach().cpu().numpy().flatten()
#     z = rnn.z.detach().cpu().numpy().flatten()

#     params_np.append({
#         "m": m,
#         "u": u,
#         "v": v,
#         "z": z
#     })

# # ---------- SAVE ----------
# data_to_save = {
#     "losses": losses,
#     "params_np": params_np,
#     "all_10d_rnn_nl": all_10d_rnn_nl,
#     "all_loss_10d_nl": all_loss_10d_nl,
#     "all_10d_rnn_nl_w": all_10d_rnn_nl_w,
#     "all_loss_10d_nl_w": all_loss_10d_nl_w,
# }

# with open("../data/fig_4.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")

# ---------- LOAD ----------
with open("../data/fig_4.pkl", "rb") as f:
    loaded_data = pickle.load(f)

losses = loaded_data["losses"]
params_np = loaded_data["params_np"]
all_10d_rnn_nl = loaded_data["all_10d_rnn_nl"]
all_loss_10d_nl = loaded_data["all_loss_10d_nl"]
all_10d_rnn_nl_w = loaded_data["all_10d_rnn_nl_w"]
all_loss_10d_nl_w = loaded_data["all_loss_10d_nl_w"]
print("Loaded successfully fig_4")

In [ ]:
all_A_overlaps = []
all_B_overlaps = [] 

for j in [0, 30_000-1, 2*30_000-1]:
    AA = [] 
    BB = [] 
    
    for ii in [0,1,2,3,4,5,6,7,8,9]:
        
        data = np.load(f'../data/memory/twin_{ii}.npz')

        params_rnn_1 = data['params_rnn_1']
        params_rnn_2 = data['params_rnn_2']
        
        AA.append(params_rnn_1[j,:])
        BB.append(params_rnn_2[j,:])
        
    all_A_overlaps.append(AA)
    all_B_overlaps.append(BB)


def evaluate_block_grouped(all_A, all_B, cols, n_runs=50, n_train_pairs=8, n_test_pairs=2, noise_std=0.3):
    
    out = []

    for i in range(3):
        A = np.array(all_A[i])[:, cols]
        B = np.array(all_B[i])[:, cols]

        assert len(A) == len(B), "A and B must have the same number of paired samples"

        n_pairs = len(A)
        accs, cms = [], []

        if i==0:

            for run in range(n_runs):
                rng = np.random.RandomState(run)
    
                pair_idx = np.arange(n_pairs)
                rng.shuffle(pair_idx)
    
                train_pairs = pair_idx[:n_train_pairs]
                test_pairs = pair_idx[n_train_pairs:n_train_pairs + n_test_pairs]
    
                # Put both A_i and B_i together in the same split
                X_train = np.vstack([A[train_pairs], B[train_pairs]])
                y_train = np.array([0] * len(train_pairs) + [1] * len(train_pairs))
    
                X_test = np.vstack([A[test_pairs], B[test_pairs]])
                y_test = np.array([0] * len(test_pairs) + [1] * len(test_pairs))
    
                # optional noise only on train
                X_train = X_train + noise_std * rng.randn(*X_train.shape)
    
                clf = make_pipeline(
                    StandardScaler(),
                    LogisticRegression()
                )
    
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
    
                accs.append(accuracy_score(y_test, y_pred))
                cms.append(confusion_matrix(y_test, y_pred, labels=[0, 1]))

        else:
            X = np.vstack((A, B))
            y = np.array([0] * len(A) + [1] * len(B))
    
            for run in range(n_runs):
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y,
                    train_size=n_train_pairs*2,
                    test_size=n_test_pairs*2,
                    stratify=y,
                    random_state=run
                )
    
                X_train = X_train + noise_std * np.random.randn(*X_train.shape)
    
                clf = make_pipeline(
                    StandardScaler(),
                    LogisticRegression()
                )
    
    
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
    
                accs.append(accuracy_score(y_test, y_pred))
                cms.append(confusion_matrix(y_test, y_pred, labels=[0, 1]))

        accs = np.array(accs)
        cm_sum = np.sum(cms, axis=0)
        out.append([accs.mean(), accs.std()])

    return out



In [ ]:
x_np, y_np, mask_np = flip_flop(4*t_max, dt, 1)

x = torch.from_numpy(x_np)        # (B,T,1)
y = torch.from_numpy(y_np)        # (B,T,1)
mask = torch.from_numpy(mask_np)  # (B,T,1)

B, T, _ = x.shape
h = torch.zeros(rnn.N, B)     # (N,B)

EP = -1
p = params_np[EP]

with torch.no_grad():
    rnn.m.copy_(torch.from_numpy(p["m"]).view_as(rnn.m))
    rnn.u.copy_(torch.from_numpy(p["u"]).view_as(rnn.u))
    rnn.v.copy_(torch.from_numpy(p["v"]).view_as(rnn.v))
    rnn.z.copy_(torch.from_numpy(p["z"]).view_as(rnn.z))

ys = []
for t in range(T):
    x_t = x[:, t, :].T                 # (1,B)
    y_t, h = rnn(x_t, h)               # y_t: (1,B)
    ys.append(y_t.T.unsqueeze(1))      # -> (B,1,1)
yhat = torch.cat(ys, dim=1)            # (B,T,1)

sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = all_10d_rnn_nl[EP]
eff_rnn_nl = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
h = torch.zeros((2,1))
y_pred = []                
for step in range(T):
    x_t = x[:, step, :].T                 # (1,B)
    y_hat, h = eff_rnn_nl(x_t, h)
    y_pred.append(y_hat)  

x_ = x_np.squeeze()
y_ = y_np.squeeze()
yhat_ = yhat.detach().numpy().squeeze()
y_pred = torch.stack(y_pred).squeeze().detach().numpy()
m_ = mask_np.squeeze()
t = np.arange(len(x_))

# import scipy.stats as stats

fig = plt.figure(figsize=(15, 6.4))
gs = gridspec.GridSpec(
    3, 3,
    width_ratios=[1., 1., 1.],
    height_ratios=[0.7, 0.7, 2],
    # hspace=0.5
)

# Left column
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)

# Middle column
ax2 = fig.add_subplot(gs[0:2, 1])

# Right column container
ax3 = fig.add_subplot(gs[0:2, 2])

ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])
ax6 = fig.add_subplot(gs[2, 2])

# ---------------- a: flip-flop task ----------------
ax0.plot(dt*t, x_, alpha=.8, lw=3, label=r'$\text{Input}$')
ax0.legend(loc='upper left', frameon=False, fontsize=14, ncols=1, handletextpad=0.2)

ax1.plot(dt*t, y_, color='k', alpha=.9, lw=3, label=r'$\text{Target}$')
ax1.plot(dt*t, yhat_, color='#0085C7', alpha=.7, lw=3, label=r'$\text{HighD RNN}$')
ax1.plot(dt*t, y_pred, color='#DF0024', ls='--', lw=2)


line0 = Line2D([0], [0], color='k', alpha=.9, lw=3)
line1 = Line2D([0], [0], color='#0085C7', lw=2)
line2 = Line2D([0], [0], color='#DF0024', lw=2, ls='--')

ax1.legend(
    [line0, (line1, line2)],
    [r'$\text{Target}$', r'$\text{RNN}$'],
    handler_map={tuple: HandlerTuple(ndivide=None)},
    frameon=False,
    fontsize=15,
    loc='upper left',
    ncols=1,
    bbox_to_anchor=(0, 1),
    handletextpad=0.2
)

ax0.set_ylabel(r'$\text{In}$', size=16, labelpad=-2)
ax0.set_yticks([-1, 1])
ax0.set_ylim([-1.2, 1.2])
ax0.set_yticklabels([])
ax0.set_title(r'$\text{Flip-flop task}$', size=18)
ax0.tick_params(labelbottom=False)

ax1.set_ylabel(r'$\text{Out}$', size=16, labelpad=-2)
ax1.set_xlabel(r'$\text{Time (s)}$', size=16, labelpad=-6)
ax1.set_yticks([-1, 1])
ax1.set_ylim([-1.2, 1.2])
ax1.set_yticklabels([])
ax1.set_xticks([10, 60])

# ---------------- b: loss curve ----------------
ax2.plot(losses[:2000], color='#0085C7', alpha=1, lw=3.5, label=r'$\theta\,\text{-space}$')
ax2.plot(all_loss_10d_nl[:2000], color='#DF0024', ls='--', lw=2, label=r'$\sigma\,\text{-space}\,(G (\theta))$')
ax2.plot(all_loss_10d_nl_w[:2000], color='k', ls='--', lw=2, label=r'$\sigma\,\text{-space}\,(\text{directly})$')

ax2.set_xticks([200, 1800])
ax2.set_yticks([0.0, 0.2])
ax2.xaxis.set_minor_locator(NullLocator())
ax2.set_xlabel(r'$\text{Epoch}$', size=16, labelpad=-6)
ax2.set_ylabel(r'$\text{Loss}$', size=16, labelpad=0)
ax2.set_title(r'$\text{Loss curve}$', size=18)
ax2.legend(loc='lower left', frameon=False, fontsize=14, handletextpad=0.2)

# ---------------- c: compact panel inside ax3 ----------------

ax3.set_axis_off()
lab = [r'$m$', r'$u$', r'$v$', r'$z$']

m_gf, u_gf, v_gf, z_gf = m,u,v,z = params_np[EP].values()
allp = [m_gf, u_gf, v_gf, z_gf]

# Tighter layout inside ax3
left, right = 0.05, 0.01
bottom, top = 0.01, 0.05
wgap, hgap = 0.05, 0.05

w = (1 - left - right - 3 * wgap) / 4
h = (1 - bottom - top - hgap) / 2

small_axes = []
for r in range(2):
    for c in range(4):
        x0_in = left + c * (w + wgap)
        y0_in = 1 - top - (r + 1) * h - r * hgap
        a = ax3.inset_axes([x0_in, y0_in, w, h])
        small_axes.append(a)

# Row 1: histograms
for i in range(4):
    a = small_axes[i]
    sns.histplot( ax=a, data=allp[i], bins=30,
        kde=True,
        alpha=.35,
        stat='count',
        color='#0085C7'
    )
    a.set_title(lab[i], size=20, pad=14)
    a.set_ylabel('')
    a.set_xlabel('')
    a.set_xticks([-5, 5])
    a.set_xlim(-7,7)
    a.set_yticks([20, 80])
    a.set_ylim(0,100)
    a.tick_params(labelsize=16, width=2, length=5, direction='in')

# Row 2: QQ plots
for i in range(4):
    a = small_axes[i + 4]
    (osm1, osr1), (slope1, intercept1, r1) = stats.probplot(allp[i], dist="norm")
    a.text(x=-1.5, y=-5, s= rf'$r = {r1:.3f}$',size=12)
    a.scatter(osm1, osr1, s=12, alpha=0.7,color='#0085C7')
    a.plot(osm1, slope1 * osm1 + intercept1, lw=2, color='k', zorder=30)
    a.set_yticks([-5, 5])
    a.set_xticks([-5, 5])
    a.set_xlim(-7,7)
    a.set_ylim(-7,7)
    a.set_xlabel('')
    a.set_ylabel('')
    a.tick_params(labelsize=16, width=2, length=5, direction='in')

# Clean up labels so panel fills better
for i, a in enumerate(small_axes):
    row = i // 4
    col = i % 4

    if row == 0:
        a.set_xticklabels([])
    if col != 0:
        a.set_yticklabels([])

    sns.despine(ax=a, top=True, right=True, left=False, bottom=False)

# ---------------- subplot labels ----------------
ax0.tick_params(axis='both', labelsize=16)
ax1.tick_params(axis='both', labelsize=16)
ax2.tick_params(axis='both', labelsize=16)


small_axes[0].set_ylabel(r'$\text{Histogram}$',size=14,labelpad=4)
small_axes[4].set_ylabel(r'$\text{Q-Q plots}$',size=14,labelpad=2)


cv_axes = [ax5,ax6]
for i,noise_std in enumerate([0.0, 0.3]): 

    z  = evaluate_block_grouped(all_A_overlaps, all_B_overlaps, slice(None, 7),noise_std=noise_std)
    zz = evaluate_block_grouped(all_A_overlaps, all_B_overlaps, slice(7, None),noise_std=noise_std)

    means = np.array(z)[:, 0]
    stds = np.array(z)[:, 1]#/np.sqrt(n_runs)
    
    x = np.arange(len(means))  # x-axis indices

    cv_axes[i].errorbar(x, means, yerr=stds, fmt='o', capsize=4, color='#0085C7')
    cv_axes[i].plot(means,color='#0085C7',label=r'$\text{Visible}$')
    
    means = np.array(zz)[:, 0]
    stds = np.array(zz)[:, 1]
    

    cv_axes[i].errorbar(x, means, yerr=stds, fmt='o', capsize=4, color='#DF0024')
    cv_axes[i].plot(means,color='#DF0024',label=r'$\text{Invisible}$')
    
    labels = [r'$\text{Initial}$', r'$\text{Task A/B}$', r'$\text{Task C}$']
    l_labels = np.arange(len(labels))
    cv_axes[i].set_xticks(l_labels, labels, rotation=-0, ha='center', size=16)
    
    cv_axes[i].set_ylabel(r'$\text{CV accuracy}$',size=16)
    
    cv_axes[i].axhline(y=0.5,ls='--',color='k') # ,label=r'$\text{random}$'

    cv_axes[i].set_ylim(-0.1,1.1)
    cv_axes[i].set_yticks([0,0.5,1])    
    cv_axes[i].tick_params(axis='both', labelsize=16)
    
    if i==0:
        cv_axes[i].legend(frameon=False, fontsize=15,loc='lower right')

    if i==0:
        cv_axes[i].set_title(r'$\text{No noise}$',size=18)
    else:
        cv_axes[i].set_title(r'$\text{With noise}$',size=18)

plt.tight_layout()
pos3 = ax3.get_position()
dx = 0.02
ax3.set_position([pos3.x0 - dx, pos3.y0, pos3.width + dx, pos3.height])

pos0 = ax0.get_position()
ax0.set_position([pos0.x0, pos0.y0-0.04, pos0.width , pos0.height+0.04])

pos1 = ax1.get_position()
ax1.set_position([pos1.x0, pos1.y0, pos1.width , pos1.height+0.08])
ax4.set_xticks([])
ax4.set_yticks([])



ax0.text(
    -0.02, 1.2, r'($\mathbf{a}$)',
    transform=ax0.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

ax2.text(
    -0.1, 1.1, r'($\mathbf{b}$)',
    transform=ax2.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

ax3.text(
    -0.1, 1.1, r'($\mathbf{c}$)',
    transform=ax3.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

ax4.text(
    -0.02, 1.13, r'($\mathbf{d}$)',
    transform=ax4.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)


ax5.text(
    -0.1, 1.13, r'($\mathbf{e}$)',
    transform=ax5.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

ax6.text(
    -0.16, 1.13, r'($\mathbf{f\,}$)',
    transform=ax6.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

sns.despine(ax=ax0)
sns.despine(ax=ax1)
sns.despine(ax=ax2)
sns.despine(ax=ax4)
sns.despine(ax=ax5)
sns.despine(ax=ax6)
sns.despine(ax=ax4, top=True, right=True, left=True, bottom=True)

# plt.savefig('../figures/fig_4.pdf', bbox_inches='tight')
plt.show()



In [ ]:
ii = 9
data = np.load(f'../data/memory/twin_{ii}.npz')

loss_rnn_1 = data['loss_rnn_1']
loss_rnn_2 = data['loss_rnn_2']
params_rnn_1 = data['params_rnn_1']
params_rnn_2 = data['params_rnn_2']

num_epochs = int(len(loss_rnn_1)/2)

fig, ax = plt.subplots(2, 3, figsize=(5.9, 3), sharex=True, sharey=True, width_ratios=[0.5, 1, 1])
ax = ax.flatten()

ax[1].set_ylabel(r"$\text{Task A}$",size=14)
ax[4].set_ylabel(r"$\text{Task B}$",size=14)

sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_1[num_epochs-1, :]
student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

x, y, _ = flip_flop(
    t_max, dt, batch_size=1,
    stim_range=(1.0, 2.0),
    input_amp=2.0,
    target_amp=1.0,
    use_torch=True
)

t_ = dt * np.arange(x.shape[1])
B, T, _ = x.shape


h = torch.zeros(2, 1)
y_pred = []
for t in range(T):
    x_t = x[:, t, :].T
    y_t, h = student_rnn_1(x_t, h)
    y_pred.append(y_t.T.unsqueeze(1))
y_pred = torch.cat(y_pred, dim=1)

ax[1].plot(t_[:-40], y[0, :-40, 0].detach().numpy(), color="k", lw=3, label=r'$\text{Target}$')
ax[1].plot(t_[:-40], y_pred[0, :-40, 0].detach().numpy(), color="#0085C7", ls='--', label=r'$\text{RNN 1}$')

sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_2[num_epochs-1, :]
student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

test_coherences = [-14, -4,4 , 14]
norm = plt.Normalize(vmin=-16, vmax=16)
cmap = cm.RdBu

for coh in test_coherences:
    x, y, _, _ = decision_making(
        t_max, dt, 1,
        coherences=[coh],
        noise_std=0.03,
        target_amp=1.0,
        fixed_coh_max=16.0,
        use_torch=True
    )

    color = cmap(norm(coh))
    t_dm = dt * np.arange(x.shape[1])



    T = x.shape[1]
    h1 = torch.zeros(2, 1)
    y_pred1 = []

    with torch.no_grad():
        for t in range(T):
            x_t = x[:, t, :].T
            yt1, h1 = student_rnn_2(x_t, h1)
            y_pred1.append(yt1)

    y_p1 = torch.cat(y_pred1, dim=1).squeeze().numpy()
    y_target = y[0, :, 0].numpy()

    ax[4].plot(t_dm[:-40], y_target[:-40],  color="k", lw=3,)
    ax[4].plot(t_dm[:-40], y_p1[:-40], color="#009F3D", ls='--', label=r'$\text{RNN 2}$')


sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_1[-1, :]
student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_2[-1, :]
student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

target_sig_zm = 0.5
target_sig_zu = 2.3
target_sig_vm = 2.0
target_sig_vu = 1.5
target_sig_mu = 1.6
target_mm = 1.8
target_uu = 2.2

pulse_scale = 2
noise_scale = 1.5
mean = 0

target_overlaps = [target_sig_zm, target_sig_zu, target_sig_vm, target_sig_vu, target_sig_mu, target_mm, target_uu]
teacher_eff_rnn = effective_rnn_from_scaler_erf(target_sig_zm, target_sig_zu, target_sig_vm, target_sig_vu, target_sig_mu, target_mm, target_uu, dt=dt)
compiled_teacher = torch.compile(teacher_eff_rnn)

x_b, _, _ = flip_flop(t_max, dt, 1, use_torch=True)
x = (noise_scale * torch.randn((T, 1))) + (pulse_scale * x_b[:, :, 0].T) + mean
t_c = dt * np.arange(T)

h_teacher = torch.zeros(2, 1)
h_student = torch.zeros(2, 1)
h_student2 = torch.zeros(2, 1)

with torch.no_grad():
    y_target_list = []
    for step in range(T):
        y_hat_t, h_teacher = teacher_eff_rnn(x[step], h_teacher)
        y_target_list.append(y_hat_t)
    y_target = torch.stack(y_target_list).squeeze()

y_pred_list = []
for step in range(T):
    y_hat_s, h_student = student_rnn_1(x[step], h_student)
    y_pred_list.append(y_hat_s)
y_pred = torch.stack(y_pred_list).squeeze()

y_pred_list2 = []
for step in range(T):
    y_hat_s, h_student2 = student_rnn_2(x[step], h_student2)
    y_pred_list2.append(y_hat_s)
y_pred2 = torch.stack(y_pred_list2).squeeze()

ax[2].plot(t_c, y_target.detach().numpy(), color="k", lw=3, label=r'$\text{Target}$')
ax[5].plot(t_c, y_target.detach().numpy(), color="k", lw=3, label=r'$\text{Target}$')
ax[2].plot(t_c, y_pred.detach().numpy(), color="#0085C7", ls='--', label=r'$\text{RNN 1}$')
ax[5].plot(t_c, y_pred2.detach().numpy(), color="#009F3D", ls='--', label=r'$\text{RNN 2}$')

for i in range(6):
    sns.despine(ax=ax[i], top=True, right=True, left=True, bottom=True)
    ax[i].set_xticks([])
    ax[i].set_yticks([])


arrow = FancyArrowPatch(
    (0.08, 0.01), (0.92, 0.01),
    transform=fig.transFigure,
    arrowstyle='-|>',
    lw=1.8,
    color='k',
    mutation_scale=15,
    clip_on=False
)
fig.add_artist(arrow)

fig.text(0.3,-0.08,s=r'$\text{Epoch (training time)}$',size=18,)
ax[2].text(-3,-2,r"$\text{Task C}$",size=15,rotation=90)


fig.text(0.11,0.045,s=r'$\text{Initial}$',size=15,)
fig.text(0.41,0.045,s=r'$\text{A/B}$',size=15,)
fig.text(0.765,0.045,s=r'$\text{C}$',size=15,)

fig.text(0.3,0.93,s=r'$\text{A/B} \rightarrow \text{C } \text{protocol}$',size=20,)


y_dot = 0.01  # align with arrow height

for x_tick in [0.16, 0.45, 0.78]:
    dot = plt.Line2D(
        [x_tick], [y_dot],
        transform=fig.transFigure,
        marker='o',
        markersize=5,
        color='k',
        linestyle='None',
        clip_on=False
    )
    fig.add_artist(dot)

# sns.despine()
# plt.tight_layout()
# plt.subplots_adjust(hspace=0.0)
# plt.savefig('../figures/fig_4d.pdf', bbox_inches='tight')
plt.show()

In [ ]:
N = 600
dt = 0.025
t_max = 20
batch_size = 10
num_epochs = 2_000

rnn_gf   = low_rank_rnn(N=N, scale=1, rank=1,  phi='erf', dt=dt, tau=1)
rnn_gd   = low_rank_rnn(N=N, scale=1, rank=1,  phi='erf', dt=dt, tau=1)
rnn_adam = low_rank_rnn(N=N, scale=1, rank=1,  phi='erf', dt=dt, tau=1)


# # ---------- TRAIN ----------
# m,u,v,z = initialize_vectors_balanced(N, 1)
# with torch.no_grad():     
#     rnn_gf.m.data.copy_(m)
#     rnn_gf.u.data.copy_(u)
#     rnn_gf.v.data.copy_(v)
#     rnn_gf.z.data.copy_(z)

# with torch.no_grad():     
#     rnn_gd.m.data.copy_(m)
#     rnn_gd.u.data.copy_(u)
#     rnn_gd.v.data.copy_(v)
#     rnn_gd.z.data.copy_(z)

# with torch.no_grad():     
#     rnn_adam.m.data.copy_(m)
#     rnn_adam.u.data.copy_(u)
#     rnn_adam.v.data.copy_(v)
#     rnn_adam.z.data.copy_(z)



# lr = 5e-2
# optimizer = optim.SGD([p for p in rnn_gf.parameters() if p.requires_grad], lr=lr*N)
# losses_gf = []
# params_gf = []

# for step in range(num_epochs+1):    
    
#     params_gf.append(deepcopy(rnn_gf))
#     x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(4.0,10.0), input_amp=1.0, target_amp=0.5, use_torch=True) 
#     B, T, _ = x.shape
#     h = torch.zeros(rnn_gf.N, B)     # (N,B)

#     ys = []
#     for t in range(T):
#         x_t = x[:, t, :].T                 # (1,B)
#         y_t, h = rnn_gf(x_t, h)               # y_t: (1,B)
#         ys.append(y_t.T.unsqueeze(1))      # -> (B,1,1)
        
#     yhat = torch.cat(ys, dim=1)            # (B,T,1)
#     _loss = masked_mse(yhat, y, mask)
#     losses_gf.append(_loss.item())
    
#     optimizer.zero_grad()
#     _loss.backward()
#     optimizer.step()

#     if step % 500 == 0 or step == 1:
#         with torch.no_grad():
#             _mse = _loss.item()
#         print(f"step {step:5d} | masked mse {_mse:.4f}")

# plt.plot(losses_gf)

# lr = 5e-1
# optimizer = optim.SGD([p for p in rnn_gd.parameters() if p.requires_grad], lr=lr*N)
# losses_gd = []
# params_gd = []

# for step in range(num_epochs+1):
    
#     params_gd.append(deepcopy(rnn_gd))

#     x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(4.0,10.0), input_amp=1.0, target_amp=0.5, use_torch=True) 
#     B, T, _ = x.shape
#     h = torch.zeros(rnn_gd.N, B)     
    
#     ys = []
#     for t in range(T):
#         x_t = x[:, t, :].T                 
#         y_t, h = rnn_gd(x_t, h)            
#         ys.append(y_t.T.unsqueeze(1))      
        
#     yhat = torch.cat(ys, dim=1)            
#     _loss = masked_mse(yhat, y, mask)
    
#     losses_gd.append(_loss.item())
    
#     optimizer.zero_grad()
#     _loss.backward()
#     optimizer.step()

#     if step % 500 == 0 or step == 1:
#         with torch.no_grad():
#             _mse = _loss.item()
#         print(f"step {step:5d} | masked mse {_mse:.4f}")
    
# plt.plot(losses_gd)

# lr = 5e-3
# optimizer = optim.Adam([p for p in rnn_adam.parameters() if p.requires_grad], lr=lr)
# losses_adam = []
# params_adam = []

# for step in range(num_epochs+1):
    
#     params_adam.append(deepcopy(rnn_adam))

#     x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(4.0,10.0), input_amp=1.0, target_amp=0.5, use_torch=True) 
#     B, T, _ = x.shape
#     h = torch.zeros(rnn_adam.N, B)     
    
#     ys = []
#     for t in range(T):
#         x_t = x[:, t, :].T                 
#         y_t, h = rnn_adam(x_t, h)          
#         ys.append(y_t.T.unsqueeze(1))      
        
#     yhat = torch.cat(ys, dim=1)            
#     _loss = masked_mse(yhat, y, mask)
    
#     losses_adam.append(_loss.item())
    
#     optimizer.zero_grad()
#     _loss.backward()
#     optimizer.step()

#     if step % 500 == 0 or step == 1:
#         with torch.no_grad():
#             _mse = _loss.item()
#         print(f"step {step:5d} | masked mse {_mse:.4f}")
    
# plt.plot(losses_adam)


# params_gf_np = []
# params_gd_np = []
# params_adam_np = []

# for ep in range(2000):
    
#     rnn_gf = params_gf[ep]
#     m = rnn_gf.m.detach().cpu().numpy().flatten()
#     u = rnn_gf.u.detach().cpu().numpy().flatten()
#     v = rnn_gf.v.detach().cpu().numpy().flatten()
#     z = rnn_gf.z.detach().cpu().numpy().flatten()

#     params_gf_np.append({
#         "m": m,
#         "u": u,
#         "v": v,
#         "z": z
#     })

#     rnn_gd = params_gd[ep]
#     m = rnn_gd.m.detach().cpu().numpy().flatten()
#     u = rnn_gd.u.detach().cpu().numpy().flatten()
#     v = rnn_gd.v.detach().cpu().numpy().flatten()
#     z = rnn_gd.z.detach().cpu().numpy().flatten()

#     params_gd_np.append({
#         "m": m,
#         "u": u,
#         "v": v,
#         "z": z
#     })

#     rnn_adam = params_adam[ep]
#     m = rnn_adam.m.detach().cpu().numpy().flatten()
#     u = rnn_adam.u.detach().cpu().numpy().flatten()
#     v = rnn_adam.v.detach().cpu().numpy().flatten()
#     z = rnn_adam.z.detach().cpu().numpy().flatten()

#     params_adam_np.append({
#         "m": m,
#         "u": u,
#         "v": v,
#         "z": z
#     })

# # ---------- SAVE ----------
# data_to_save = {
#     "losses_gf": losses_gf,
#     "params_gf_np": params_gf_np,
#     "losses_gd": losses_gd,
#     "params_gd_np": params_gd_np,
#     "losses_adam": losses_adam,
#     "params_adam_np": params_adam_np,
# }

# with open("../data/fig_8.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")


# ---------- LOAD ----------
with open("../data/fig_8.pkl", "rb") as f:
    loaded_data = pickle.load(f)

losses_gf = loaded_data["losses_gf"]
params_gf_np = loaded_data["params_gf_np"]
losses_gd = loaded_data["losses_gd"]
params_gd_np = loaded_data["params_gd_np"]
losses_adam = loaded_data["losses_adam"]
params_adam_np = loaded_data["params_adam_np"]
print("Loaded successfully fig_8")


In [ ]:
EP = -1

runs = {
    "gf": params_gf_np,
    "gd": params_gd_np,
    "adam": params_adam_np,
}

rnn = low_rank_rnn(N=N, scale=1, rank=1,  phi='erf', dt=dt, tau=1)

results = {}
x_np, y_np, mask_np = flip_flop(3*t_max, dt, 1)

x = torch.from_numpy(x_np)        # (B,T,1)
y = torch.from_numpy(y_np)        # (B,T,1)
mask = torch.from_numpy(mask_np)  # (B,T,1)
target = y_np.squeeze()
t_ = np.arange(len(x_np.squeeze()))
B, T, _ = x.shape

for name, params in runs.items():
    m = params[EP]['m']
    u = params[EP]['u']
    v = params[EP]['v']
    z = params[EP]['z']

    with torch.no_grad():
        rnn.m.copy_(torch.as_tensor(m.reshape(-1, 1), dtype=rnn.m.dtype, device=rnn.m.device))
        rnn.u.copy_(torch.as_tensor(u.reshape(-1, 1), dtype=rnn.u.dtype, device=rnn.u.device))
        rnn.v.copy_(torch.as_tensor(v.reshape(-1, 1), dtype=rnn.v.dtype, device=rnn.v.device))
        rnn.z.copy_(torch.as_tensor(z.reshape(-1, 1), dtype=rnn.z.dtype, device=rnn.z.device))

    sig_zm = z @ m / N
    sig_zu = z @ u / N
    sig_vm = v @ m / N
    sig_vu = v @ u / N
    sig_mu = m @ u / N
    sig_zv = z @ v / N

    mm = m @ m / N
    uu = u @ u / N
    vv = v @ v / N
    zz = z @ z / N

    eff_rnn = effective_rnn_from_scaler_erf(
        sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt
    )

    h = torch.zeros(rnn.N, B)
    y_high_d = []
    for t in range(T):
        x_t = x[:, t, :].T
        y_t, h = rnn(x_t, h)
        y_high_d.append(y_t.T.unsqueeze(1))
    y_high_d = torch.cat(y_high_d, dim=1).squeeze().detach().numpy()

    h = torch.zeros((2, 1))
    y_low_d = []
    for t in range(T):
        x_t = x[:, t, :].T
        y_hat, h = eff_rnn(x_t, h)
        y_low_d.append(y_hat)
    y_low_d = torch.stack(y_low_d).squeeze().detach().numpy()

    results[name] = {
        "high": y_high_d,
        "low": y_low_d,
    }

    # plt.plot(dt * t_, target, color='k', alpha=1, lw=2, label=r'$\text{Target}$')
    # plt.plot(dt * t_, y_high_d, color='#0085C7', alpha=1, lw=2, label=r'$\text{HighD RNN}$')
    # plt.plot(dt * t_, y_low_d, color='#DF0024', ls='--', lw=2, label=r'$\text{Eff. RNN}$')
    # plt.title(name.upper())
    # plt.tight_layout()
    # sns.despine()
    # plt.show()

y_high_d_gf   = results["gf"]["high"]
y_low_d_gf    = results["gf"]["low"]
y_high_d_gd   = results["gd"]["high"]
y_low_d_gd    = results["gd"]["low"]
y_high_d_adam = results["adam"]["high"]
y_low_d_adam  = results["adam"]["low"]

m_gf, u_gf, v_gf, z_gf = params_gf_np[EP]['m'], params_gf_np[EP]['u'], params_gf_np[EP]['v'], params_gf_np[EP]['z']
m_gd, u_gd, v_gd, z_gd = params_gd_np[EP]['m'], params_gd_np[EP]['u'], params_gd_np[EP]['v'], params_gd_np[EP]['z']
m_adam, u_adam, v_adam, z_adam = params_adam_np[EP]['m'], params_adam_np[EP]['u'], params_adam_np[EP]['v'], params_adam_np[EP]['z']

titles = [r'$\text{GF}$', r'$\text{GD}$', r'$\text{Adam}$']
trace_data = [
    (y_high_d_gf, y_low_d_gf),
    (y_high_d_gd, y_low_d_gd),
    (y_high_d_adam, y_low_d_adam),
]

hist_data = [
    [m_gf, u_gf, v_gf, z_gf],
    [m_gd, u_gd, v_gd, z_gd],
    [m_adam, u_adam, v_adam, z_adam],
]

row_colors = ['#0085C7', '#DF0024', '#009F3D']
lab = [r'$m$', r'$u$', r'$v$', r'$z$']

fig, ax = plt.subplots(
    3, 5, figsize=(14, 7),
    gridspec_kw={'width_ratios': [2, 1, 1, 1, 1]},
    sharex='col'
)

for r in range(3):
    # -------------------------
    # Left column: trace plots
    # -------------------------
    ax[r, 0].plot(dt*t_, target, color='gray',lw=3, alpha=0.8 ,label=r'$\text{Target}$')
    ax[r, 0].plot(dt*t_, trace_data[r][0], color=row_colors[r], lw=3, label=r'$\text{High-D RNN}$')
    ax[r, 0].plot(dt*t_, trace_data[r][1], color='k', ls='--', lw=2, label=r'$\text{Eff. RNN}$')

    # ax[r, 0].set_title(titles[r], fontsize=18)
    ax[r, 0].set_ylim(-0.74, 0.74)
    ax[r, 0].set_xticks([10, 70])
    ax[r, 0].set_yticks([-.5, .5])
    ax[r, 0].xaxis.set_major_locator(MaxNLocator(nbins=4))

    
    ax[0, 0].set_ylabel(r'$\text{Output}$', size=16, labelpad=-10)
    ax[1, 0].set_ylabel(r'$\text{Output}$', size=16, labelpad=-10)
    ax[2, 0].set_ylabel(r'$\text{Output}$', size=16, labelpad=-10)
    
    if r == 2:
        ax[r, 0].set_xlabel(r'$\text{Time-step}$', size=16)

    # -------------------------
    # Right columns: histograms
    # -------------------------
    for c in range(4):
        sns.histplot(
            ax=ax[r, c+1],
            data=hist_data[r][c],
            bins=15,
            kde=True,
            alpha=.4,
            color=row_colors[r],
            stat="density"
        )

        
        ax[0, c+1].set_title(lab[c], size=20)
        ax[1, c+1].set_title(lab[c], size=20)
        ax[2, c+1].set_title(lab[c], size=20)

        ax[r, c+1].set_ylabel('')
        ax[r, c+1].yaxis.set_major_locator(MaxNLocator(nbins=2))

# Create legend line styles
line0 = Line2D([0], [0],  color='gray' ,alpha=.8, lw=3)                     # Target
line1 = Line2D([0], [0], color='k', lw=2)               # Full
# line2 = Line2D([0], [0], color='#DF0024', lw=2)      # Eff
# line3 = Line2D([0], [0], color='#009F3D', lw=2)
line4 = Line2D([0], [0], color='k', lw=2, ls='--')

ax[0,0].legend(
    [line0, line1, line4],
    [r'$\text{Target}$', r'$\text{RNN}$', r'$\text{Eff}.$'],
    handler_map={tuple: HandlerTuple(ndivide=None)},
    frameon=False,
    fontsize=14,
    loc='upper left',
    ncols=3,
    bbox_to_anchor=(-.1, 1.24), 
)

fig.text(0.65, .94, r'$\text{GF}$', ha='center', fontsize=20, color='#0085C7')
fig.text(0.65, 0.64, r'$\text{GD}$', ha='center', fontsize=20, color="#DF0024")
fig.text(0.65, 0.34, r'$\text{Adam}$', ha='center', fontsize=20, color="#009F3D")


subplot_labels = [r'($\mathbf{a}$)', r'($\mathbf{b}$)', r'($\mathbf{c}$)']
for i, label in enumerate(subplot_labels):
    ax[i,0].text(-0.15, 1.15, label, transform=ax[i,0].transAxes,
               fontsize=18, fontweight='bold', va='top', ha='right', font="Arial")


sns.despine()
plt.tight_layout()
plt.subplots_adjust(wspace=0.4, hspace=0.4)
# plt.savefig('../figures/fig_8.pdf', bbox_inches='tight')
plt.show()
    

In [ ]:
dt = 0.05
t_max = 20
T = int(t_max / dt) 
num_epochs = 30_000
batch_size = 128

target_sig_zm = 0.5
target_sig_zu = 2.3
target_sig_vm = 2.0
target_sig_vu = 1.5
target_sig_mu = 1.6
target_mm = 1.8
target_uu = 2.2

pulse_scale = 2
noise_scale = 1.5
mean = 0

target_overlaps = [target_sig_zm, target_sig_zu, target_sig_vm, target_sig_vu, target_sig_mu, target_mm, target_uu]
teacher_eff_rnn = effective_rnn_from_scaler_erf(target_sig_zm, target_sig_zu, target_sig_vm, target_sig_vu, target_sig_mu, target_mm, target_uu, dt=dt)
compiled_teacher = torch.compile(teacher_eff_rnn)

for ii in tqdm(range(10)):

    params_rnn_1, loss_rnn_1 = [], []
    params_rnn_2, loss_rnn_2 = [], []

    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, sig_zv = np.abs(np.random.normal(0,0.4,6))
    mm, uu, vv, zz = np.random.uniform(0.5,2.0,4)

    ################## A ##################
    student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    compiled_student_1 = torch.compile(student_rnn_1)

    lr = 0.01
    flag = 1
    for epoch in range(num_epochs):
        
        # log parameters 
        params_rnn_1.append([
            float(sig_zm), float(sig_zu), float(sig_vm), float(sig_vu),
            float(sig_mu), float(mm), float(uu), float(sig_zv), float(vv), float(zz)
        ])
    
        # sample a batch 
        x, y, mask = flip_flop(t_max, dt, batch_size, stim_range=(1.0,2.0), input_amp=2.0, target_amp=1.0, use_torch=True) 
        B, T, _ = x.shape
        h = torch.zeros(2, B)    
    
        # forward pass
        ys = []
        for t in range(T):
            x_t = x[:, t, :].T                 
            y_t, h = compiled_student_1(x_t, h)     
            ys.append(y_t.T.unsqueeze(1))      
    
        # calc loss
        yhat = torch.cat(ys, dim=1)            
        _loss = masked_mse(yhat, y, mask)
    
        # log loss
        loss_rnn_1.append(_loss.item())
        
        # backward pass (to calc gradients)   
        student_rnn_1.zero_grad()
        _loss.backward()

        if flag and _loss.item() < 0.015:
            lr /= 5
            flag = 0
            print('ff | '+ f'ep: {epoch} | ' +  f'run {ii}')
        
        with torch.no_grad():
            # extract gradients 
            dzm, dzu = student_rnn_1.zm.grad, student_rnn_1.zu.grad
            dvm, dvu = student_rnn_1.vm.grad, student_rnn_1.vu.grad
            dmu = student_rnn_1.mu.grad
            dmm, duu = student_rnn_1.mm.grad, student_rnn_1.uu.grad
        
            # Apply updates
            update_scale = lr * 1.0 
            
            sig_zm -= update_scale * ( (mm + zz) * dzm + sig_mu * dzu + sig_zv * dvm + sig_zu * dmu + 2.0 * sig_zm * dmm )
            sig_zu -= update_scale * ( sig_mu * dzm + (uu + zz) * dzu + sig_zv * dvu + sig_zm * dmu + 2.0 * sig_zu * duu )
            sig_vm -= update_scale * ( (mm + vv) * dvm + sig_mu * dvu + sig_zv * dzm + sig_vu * dmu + 2.0 * sig_vm * dmm )
            sig_vu -= update_scale * ( sig_mu * dvm + (uu + vv) * dvu + sig_zv * dzu + sig_vm * dmu + 2.0 * sig_vu * duu )
            
            sig_mu -= update_scale * ( sig_zu * dzm + sig_zm * dzu + sig_vu * dvm + sig_vm * dvu + (mm + uu) * dmu + 2.0 * sig_mu * (dmm + duu) )
            sig_zv -= update_scale * ( sig_vm * dzm + sig_vu * dzu + sig_zm * dvm + sig_zu * dvu )
            
            mm -= update_scale * ( 2.0 * (sig_zm * dzm + sig_vm * dvm + sig_mu * dmu + 2.0 * mm * dmm ) )
            uu -= update_scale * ( 2.0 * (sig_zu * dzu + sig_vu * dvu + sig_mu * dmu + 2.0 * uu * duu ) )
            vv -= update_scale * ( 2.0 * (sig_vm * dvm + sig_vu * dvu) ) 
            zz -= update_scale * ( 2.0 * (sig_zm * dzm + sig_zu * dzu) )
        
            # --- Update the actual parameters in the student model ---
            student_rnn_1.zm.copy_(torch.as_tensor(sig_zm))
            student_rnn_1.zu.copy_(torch.as_tensor(sig_zu))
            student_rnn_1.vm.copy_(torch.as_tensor(sig_vm))
            student_rnn_1.vu.copy_(torch.as_tensor(sig_vu))
            student_rnn_1.mu.copy_(torch.as_tensor(sig_mu))
            student_rnn_1.mm.copy_(torch.as_tensor(mm))
            student_rnn_1.uu.copy_(torch.as_tensor(uu))

    ################## B ##################
    # twin
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = np.array(params_rnn_1)[0,:]
    
    # random  
    # sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, sig_zv = np.abs(np.random.normal(0,0.4,6))
    # mm, uu, vv, zz = np.random.uniform(0.5,2.0,4)

    student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    compiled_student_2 = torch.compile(student_rnn_2)

    lr = 0.01
    flag = 1
    for epoch in range(num_epochs):
        
        # log parameters 
        params_rnn_2.append([
            float(sig_zm), float(sig_zu), float(sig_vm), float(sig_vu),
            float(sig_mu), float(mm), float(uu), float(sig_zv), float(vv), float(zz)
        ])
    
        # sample a batch 
        x, y, mask, _ = decision_making(t_max, dt, batch_size, coherences=[-16, -8, -2, 2, 8, 16],
                                    noise_std=0.05, target_amp=1.0, use_torch=True) 
        B, T, _ = x.shape
        h = torch.zeros(2, B)    
    
        # forward pass
        ys = []
        for t in range(T):
            x_t = x[:, t, :].T                 
            y_t, h = compiled_student_2(x_t, h)     
            ys.append(y_t.T.unsqueeze(1))      
    
        # calc loss
        yhat = torch.cat(ys, dim=1)            
        _loss = masked_mse(yhat, y, mask)
    
        # log loss
        loss_rnn_2.append(_loss.item())
        
        # backward pass (to calc gradients)   
        student_rnn_2.zero_grad()
        _loss.backward()

        if flag and _loss.item() < 0.015:
            lr /= 5
            flag = 0
            print('dm | '+ f'ep: {epoch} | ' +  f'run {ii}')
    
        with torch.no_grad():
            # extract gradients             
            dzm, dzu = student_rnn_2.zm.grad, student_rnn_2.zu.grad
            dvm, dvu = student_rnn_2.vm.grad, student_rnn_2.vu.grad
            dmu = student_rnn_2.mu.grad
            dmm, duu = student_rnn_2.mm.grad, student_rnn_2.uu.grad
        
            # Apply updates
            update_scale = lr * 1.0 
            
            sig_zm -= update_scale * ( (mm + zz) * dzm + sig_mu * dzu + sig_zv * dvm + sig_zu * dmu + 2.0 * sig_zm * dmm )
            sig_zu -= update_scale * ( sig_mu * dzm + (uu + zz) * dzu + sig_zv * dvu + sig_zm * dmu + 2.0 * sig_zu * duu )
            sig_vm -= update_scale * ( (mm + vv) * dvm + sig_mu * dvu + sig_zv * dzm + sig_vu * dmu + 2.0 * sig_vm * dmm )
            sig_vu -= update_scale * ( sig_mu * dvm + (uu + vv) * dvu + sig_zv * dzu + sig_vm * dmu + 2.0 * sig_vu * duu )
            
            sig_mu -= update_scale * ( sig_zu * dzm + sig_zm * dzu + sig_vu * dvm + sig_vm * dvu + (mm + uu) * dmu + 2.0 * sig_mu * (dmm + duu) )
            sig_zv -= update_scale * ( sig_vm * dzm + sig_vu * dzu + sig_zm * dvm + sig_zu * dvu )
            
            mm -= update_scale * ( 2.0 * (sig_zm * dzm + sig_vm * dvm + sig_mu * dmu + 2.0 * mm * dmm ) )
            uu -= update_scale * ( 2.0 * (sig_zu * dzu + sig_vu * dvu + sig_mu * dmu + 2.0 * uu * duu ) )
            vv -= update_scale * ( 2.0 * (sig_vm * dvm + sig_vu * dvu) ) 
            zz -= update_scale * ( 2.0 * (sig_zm * dzm + sig_zu * dzu) )
        
            # --- Update the actual parameters in the student model ---
            student_rnn_2.zm.copy_(torch.as_tensor(sig_zm))
            student_rnn_2.zu.copy_(torch.as_tensor(sig_zu))
            student_rnn_2.vm.copy_(torch.as_tensor(sig_vm))
            student_rnn_2.vu.copy_(torch.as_tensor(sig_vu))
            student_rnn_2.mu.copy_(torch.as_tensor(sig_mu))
            student_rnn_2.mm.copy_(torch.as_tensor(mm))
            student_rnn_2.uu.copy_(torch.as_tensor(uu))


    ################## C ##################
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = np.array(params_rnn_1)[-1,:]
    student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    compiled_student_1 = torch.compile(student_rnn_1)

    lr = 0.001
    for epoch in range(num_epochs):
        
        # log parameters 
        params_rnn_1.append([
            float(sig_zm), float(sig_zu), float(sig_vm), float(sig_vu),
            float(sig_mu), float(mm), float(uu), float(sig_zv), float(vv), float(zz)
        ])
    
        # --- Data Prep ---
        x_c, _, _ = flip_flop(t_max, dt, batch_size, use_torch=True)
        x = (noise_scale * torch.randn((T, batch_size))) + (pulse_scale * x_c[:,:,0].T) + mean
        
        h_teacher = torch.zeros(2, batch_size)
        h_student = torch.zeros(2, batch_size)
        
        # --- Teacher Pass (No Gradients) ---
        with torch.no_grad():
            y_target_list = []
            for step in range(T):
                y_hat_t, h_teacher = compiled_teacher(x[step], h_teacher)
                y_target_list.append(y_hat_t)
            y_target = torch.stack(y_target_list).squeeze()
        
        # --- Student Pass ---
        y_pred_list = []
        for step in range(T):
            y_hat_s, h_student = compiled_student_1(x[step], h_student)
            y_pred_list.append(y_hat_s)
        y_pred = torch.stack(y_pred_list).squeeze()
        
        # Vectorized Loss Calculation
        _loss = torch.mean((y_target - y_pred)**2) * (T * dt)
    
        # log loss
        loss_rnn_1.append(_loss.item())
        
        # backward pass (to calc gradients)   
        student_rnn_1.zero_grad()
        _loss.backward()
        
        with torch.no_grad():
            # extract gradients 
            dzm, dzu = student_rnn_1.zm.grad, student_rnn_1.zu.grad
            dvm, dvu = student_rnn_1.vm.grad, student_rnn_1.vu.grad
            dmu = student_rnn_1.mu.grad
            dmm, duu = student_rnn_1.mm.grad, student_rnn_1.uu.grad
        
            # Apply updates
            update_scale = lr * 1.0 
            
            sig_zm -= update_scale * ( (mm + zz) * dzm + sig_mu * dzu + sig_zv * dvm + sig_zu * dmu + 2.0 * sig_zm * dmm )
            sig_zu -= update_scale * ( sig_mu * dzm + (uu + zz) * dzu + sig_zv * dvu + sig_zm * dmu + 2.0 * sig_zu * duu )
            sig_vm -= update_scale * ( (mm + vv) * dvm + sig_mu * dvu + sig_zv * dzm + sig_vu * dmu + 2.0 * sig_vm * dmm )
            sig_vu -= update_scale * ( sig_mu * dvm + (uu + vv) * dvu + sig_zv * dzu + sig_vm * dmu + 2.0 * sig_vu * duu )
            
            sig_mu -= update_scale * ( sig_zu * dzm + sig_zm * dzu + sig_vu * dvm + sig_vm * dvu + (mm + uu) * dmu + 2.0 * sig_mu * (dmm + duu) )
            sig_zv -= update_scale * ( sig_vm * dzm + sig_vu * dzu + sig_zm * dvm + sig_zu * dvu )
            
            mm -= update_scale * ( 2.0 * (sig_zm * dzm + sig_vm * dvm + sig_mu * dmu + 2.0 * mm * dmm ) )
            uu -= update_scale * ( 2.0 * (sig_zu * dzu + sig_vu * dvu + sig_mu * dmu + 2.0 * uu * duu ) )
            vv -= update_scale * ( 2.0 * (sig_vm * dvm + sig_vu * dvu) ) 
            zz -= update_scale * ( 2.0 * (sig_zm * dzm + sig_zu * dzu) )
        
            # --- Update the actual parameters in the student model ---
            student_rnn_1.zm.copy_(torch.as_tensor(sig_zm))
            student_rnn_1.zu.copy_(torch.as_tensor(sig_zu))
            student_rnn_1.vm.copy_(torch.as_tensor(sig_vm))
            student_rnn_1.vu.copy_(torch.as_tensor(sig_vu))
            student_rnn_1.mu.copy_(torch.as_tensor(sig_mu))
            student_rnn_1.mm.copy_(torch.as_tensor(mm))
            student_rnn_1.uu.copy_(torch.as_tensor(uu))
    
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = np.array(params_rnn_2)[-1,:]
    student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    compiled_student_2 = torch.compile(student_rnn_2)

    lr = 0.001
    for epoch in range(num_epochs):    
        # log parameters 
        params_rnn_2.append([
            float(sig_zm), float(sig_zu), float(sig_vm), float(sig_vu),
            float(sig_mu), float(mm), float(uu), float(sig_zv), float(vv), float(zz)
        ])
    
        x_c, _, _ = flip_flop(t_max, dt, batch_size, use_torch=True)
        x = (noise_scale * torch.randn((T, batch_size))) + (pulse_scale * x_c[:,:,0].T) + mean
        
        h_teacher = torch.zeros(2, batch_size)
        h_student = torch.zeros(2, batch_size)
        
        # --- Teacher Pass (No Gradients) ---
        with torch.no_grad():
            y_target_list = []
            for step in range(T):
                y_hat_t, h_teacher = compiled_teacher(x[step], h_teacher)
                y_target_list.append(y_hat_t)
            y_target = torch.stack(y_target_list).squeeze()
        
        # --- Student Pass ---
        y_pred_list = []
        for step in range(T):
            y_hat_s, h_student = compiled_student_2(x[step], h_student)
            y_pred_list.append(y_hat_s)
        y_pred = torch.stack(y_pred_list).squeeze()
        
        # Vectorized Loss Calculation
        _loss = torch.mean((y_target - y_pred)**2) * (T * dt)
    
        # log loss
        loss_rnn_2.append(_loss.item())
        
        # backward pass (to calc gradients)   
        student_rnn_2.zero_grad()
        _loss.backward()
    
        with torch.no_grad():
            # extract gradients             
            dzm, dzu = student_rnn_2.zm.grad, student_rnn_2.zu.grad
            dvm, dvu = student_rnn_2.vm.grad, student_rnn_2.vu.grad
            dmu = student_rnn_2.mu.grad
            dmm, duu = student_rnn_2.mm.grad, student_rnn_2.uu.grad
        
            # Apply updates
            update_scale = lr * 1.0 
            
            sig_zm -= update_scale * ( (mm + zz) * dzm + sig_mu * dzu + sig_zv * dvm + sig_zu * dmu + 2.0 * sig_zm * dmm )
            sig_zu -= update_scale * ( sig_mu * dzm + (uu + zz) * dzu + sig_zv * dvu + sig_zm * dmu + 2.0 * sig_zu * duu )
            sig_vm -= update_scale * ( (mm + vv) * dvm + sig_mu * dvu + sig_zv * dzm + sig_vu * dmu + 2.0 * sig_vm * dmm )
            sig_vu -= update_scale * ( sig_mu * dvm + (uu + vv) * dvu + sig_zv * dzu + sig_vm * dmu + 2.0 * sig_vu * duu )
            
            sig_mu -= update_scale * ( sig_zu * dzm + sig_zm * dzu + sig_vu * dvm + sig_vm * dvu + (mm + uu) * dmu + 2.0 * sig_mu * (dmm + duu) )
            sig_zv -= update_scale * ( sig_vm * dzm + sig_vu * dzu + sig_zm * dvm + sig_zu * dvu )
            
            mm -= update_scale * ( 2.0 * (sig_zm * dzm + sig_vm * dvm + sig_mu * dmu + 2.0 * mm * dmm ) )
            uu -= update_scale * ( 2.0 * (sig_zu * dzu + sig_vu * dvu + sig_mu * dmu + 2.0 * uu * duu ) )
            vv -= update_scale * ( 2.0 * (sig_vm * dvm + sig_vu * dvu) ) 
            zz -= update_scale * ( 2.0 * (sig_zm * dzm + sig_zu * dzu) )
        
            # --- Update the actual parameters in the student model ---
            student_rnn_2.zm.copy_(torch.as_tensor(sig_zm))
            student_rnn_2.zu.copy_(torch.as_tensor(sig_zu))
            student_rnn_2.vm.copy_(torch.as_tensor(sig_vm))
            student_rnn_2.vu.copy_(torch.as_tensor(sig_vu))
            student_rnn_2.mu.copy_(torch.as_tensor(sig_mu))
            student_rnn_2.mm.copy_(torch.as_tensor(mm))
            student_rnn_2.uu.copy_(torch.as_tensor(uu))
            
    np.savez(f'../data/memory/twin_{ii}.npz',
         loss_rnn_1=loss_rnn_1,
         loss_rnn_2=loss_rnn_2,
         params_rnn_1=params_rnn_1,
         params_rnn_2=params_rnn_2)


    params_rnn_1 = np.array(params_rnn_1)
    params_rnn_2 = np.array(params_rnn_2)
    
    fig, ax = plt.subplots(1, 5, figsize=(18, 4))
    
    ax[0].plot(loss_rnn_1[:num_epochs])
    ax[0].plot(loss_rnn_2[:num_epochs])
    
    ax[0].set_xscale('log')
    ax[0].set_yscale('log')
    
    ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('Loss')
    
    ax[1].plot(loss_rnn_1[num_epochs:])
    ax[1].plot(loss_rnn_2[num_epochs:])
    
    ax[1].set_xscale('log')
    ax[1].set_yscale('log')
    
    ax[1].set_xlabel('Epoch')
    ax[1].set_ylabel('Loss')
    
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_1[num_epochs-1,:]
    student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    
    x, y, _ = flip_flop(t_max, dt, batch_size=1, stim_range=(1.0,2.0), input_amp=2.0, target_amp=1.0, use_torch=True) 
    t_ = dt*np.arange(x.shape[1])
    B, T, _ = x.shape
    h = torch.zeros(2, 1)    
    y_pred = []
    for t in range(T):
        x_t = x[:, t, :].T                 
        y_t, h = student_rnn_1(x_t, h)     
        y_pred.append(y_t.T.unsqueeze(1))      
    y_pred = torch.cat(y_pred, dim=1)  
    
    ax[2].plot(t_, y[0,:,0].detach().numpy(),lw=2, label='Target')
    ax[2].plot(t_, y_pred[0,:,0].detach().numpy(),'--k', label='Student')
    ax[2].set_xlabel("Time (s)")
    ax[2].set_ylabel('Output')
    ax[2].legend()
    
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_2[num_epochs-1,:]
    student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    
    test_coherences = [-16, -8, -4, 4, 8, 16]
    norm = plt.Normalize(vmin=-16, vmax=16)
    cmap = cm.RdYlBu_r  # Red for positive (Right), Blue for negative (Left)
    
    for coh in test_coherences:
        
        x, y, _, _ = decision_making(t_max, dt, 1, coherences=[coh], 
                                     noise_std=0.03, target_amp=1.0, 
                                     fixed_coh_max=16.0, use_torch=True) 
        
        t_ = dt * np.arange(x.shape[1])
        T = x.shape[1]
        
        h1 = torch.zeros(2, 1)  
        
        y_pred1 = []
    
        with torch.no_grad():
            for t in range(T):
                x_t = x[:, t, :].T                 
                yt1, h1 = student_rnn_2(x_t, h1)     
                y_pred1.append(yt1)
                
        y_p1 = torch.cat(y_pred1, dim=1).squeeze().numpy()
        y_target = y[0, :, 0].numpy()
        color = cmap(norm(coh))
    
        ax[3].plot(t_, y_target, color=color, linestyle='--', lw=2, alpha=0.7)
        ax[3].plot(t_, y_p1, color=color, label=f'Coh {coh}')
    
        ax[3].set_xlabel("Time (s)")
        ax[3].set_ylabel("Output")
        ax[3].axhline(0, color='k', alpha=0.2, lw=1)
        ax[3].legend()
    
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_1[-1,:]
    student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    
    sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_2[-1,:]
    student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)
    
    x_b, _, _ = flip_flop(t_max, dt, 1, use_torch=True)
    x = (noise_scale * torch.randn((T, 1))) + (pulse_scale * x_b[:,:,0].T) + mean
    
    h_teacher = torch.zeros(2, 1)
    h_student = torch.zeros(2, 1)
    h_student2 = torch.zeros(2, 1)
    
    with torch.no_grad():
        y_target_list = []
        for step in range(T):
            y_hat_t, h_teacher = compiled_teacher(x[step], h_teacher)
            y_target_list.append(y_hat_t)
        y_target = torch.stack(y_target_list).squeeze()
    
    # --- Student Pass ---
    y_pred_list = []
    for step in range(T):
        y_hat_s, h_student = compiled_student_1(x[step], h_student)
        y_pred_list.append(y_hat_s)
    y_pred = torch.stack(y_pred_list).squeeze()
    
    # --- Student Pass ---
    y_pred_list2 = []
    for step in range(T):
        y_hat_s, h_student2 = compiled_student_2(x[step], h_student2)
        y_pred_list2.append(y_hat_s)
    y_pred2 = torch.stack(y_pred_list2).squeeze()
    
    ax[4].plot(t_,y_target.detach().numpy(),'--k',lw=3,label='Target')
    ax[4].plot(t_,y_pred.detach().numpy(),'--b',label='Student 1 ')
    ax[4].plot(t_,y_pred2.detach().numpy(),'--r',label='Student 2 ')
    ax[4].set_xlabel("Time (s)")
    ax[4].set_ylabel("Output")
    ax[4].legend()
    
    plt.tight_layout()
    sns.despine()
    plt.show()
    
    fig, ax = plt.subplots(2, 5, figsize=(18, 6), sharex=True)
    ax = ax.flatten() 
    
    labels = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$',
           r'$\sigma_{mu}$', r'$||m||^2$', r'$||u||^2$',  r'$\sigma_{zv}$', 
           r'$||v||^2$', r'$||z||^2$']
    
    for i, lab in enumerate(labels):
        ax[i].plot(params_rnn_1[:, i])
        ax[i].plot(params_rnn_2[:, i])
        ax[i].axvline(x=num_epochs,color='k')
        ax[i].set_title(lab, fontsize=14)
        ax[i].set_xlabel('Epoch')
        ax[i].set_xscale('log')
        
    plt.tight_layout()
    sns.despine()
    plt.show()
    
    fig,ax = plt.subplots(1,2,figsize=(9,4))
    
    ax[0].scatter(params_rnn_1[num_epochs-1,:7],params_rnn_2[num_epochs-1,:7],s=30)
    ax[0].scatter(params_rnn_1[num_epochs-1,7:],params_rnn_2[num_epochs-1,7:],s=30)
    
    ax[1].scatter(params_rnn_1[-1,:7],params_rnn_2[-1,:7],s=30)
    ax[1].scatter(params_rnn_1[-1,7:],params_rnn_2[-1,7:],s=30)
    
    ax[0].plot([0,10],[0,10],'--k',lw=0.1)
    ax[1].plot([0,10],[0,10],'--k',lw=0.1)
    
    ax[0].set_title('A/B')
    ax[1].set_title('C')
    
    ax[0].set_xlabel('Net 1')
    ax[0].set_ylabel('Net 2')
    ax[1].set_xlabel('Net 1')
    ax[1].set_ylabel('Net 2')
    
    plt.tight_layout()
    sns.despine()
    plt.show()
        

In [ ]:
fig = plt.figure(figsize=(12, 7))
gs = gridspec.GridSpec(
    2, 3,
    figure=fig,
    height_ratios=[1.0, 1.3],
    # width_ratios=[1, 1, 1],
    # hspace=0.5,
    # wspace=0.3
)

# =========================================================
# --------------------- FIRST ROW -------------------------
# =========================================================
ax00 = fig.add_subplot(gs[0, 0])
ax01 = fig.add_subplot(gs[0, 1])
ax02 = fig.add_subplot(gs[0, 2])

ii = 9
data = np.load(f'../data/memory/twin_{ii}.npz')

loss_rnn_1 = data['loss_rnn_1']
loss_rnn_2 = data['loss_rnn_2']
params_rnn_1 = data['params_rnn_1']
params_rnn_2 = data['params_rnn_2']

ax00.plot(loss_rnn_1, label=r'$\text{RNN 1}$', color="#0085C7")
ax00.plot(loss_rnn_2, label=r'$\text{RNN 2}$', color="#DF0024") #(A} \rightarrow \text{C)}
ax00.set_yscale('log')
ax00.set_ylabel(r'$\text{Loss}$', size=17, labelpad=0)
ax00.set_title(r'$\text{Loss curve}$', size=18)
ax00.set_yticks([0.001, 0.1, 10])
ax00.set_xticks([10_000, 30_000 ,50_000])
ax00.set_xticklabels([10, 30, 50])
ax00.set_xlabel(r'$\text{Epoch} \; (\times 10^3)$', size=18, labelpad=0)
ax00.axvline(x=30_000, color='k', lw=2, ymin=0.02, ymax=0.98, ls='--')
ax00.legend(loc='upper left', frameon=False, fontsize=14)
ax00.text(x=15_000, y=0.02, s=r'$\text{B}$', size=20, zorder=30,)
ax00.text(x=15_000, y=0.001, s=r'$\text{A}$', size=20, zorder=30,)
ax00.text(x=45_000, y=0.01, s=r'$\text{C}$', size=20, zorder=30,)
ax00.yaxis.set_minor_locator(NullLocator())

ax01.scatter(params_rnn_1[num_epochs-1, :7], params_rnn_2[num_epochs-1, :7],
             s=60, color='#0085C7', label=r'$\text{Visible}$', edgecolor="k", lw=0.5)
ax01.scatter(params_rnn_1[num_epochs-1, 7:], params_rnn_2[num_epochs-1, 7:],
             s=60, color='#DF0024', label=r'$\text{Invisible}$', edgecolor="k", lw=0.5)

ax02.scatter(params_rnn_1[-1, :7], params_rnn_2[-1, :7],
             s=60, color='#0085C7', label=r'$\text{Visible}$', edgecolor="k", lw=0.5)
ax02.scatter(params_rnn_1[-1, 7:], params_rnn_2[-1, 7:],
             s=60, color='#DF0024', label=r'$\text{Invisible}$', edgecolor="k", lw=0.5)

for a in [ax01, ax02]:
    a.plot([-2, 10], [-2, 10], '--k', lw=0.5)
    a.set_xticks([0, 5, 10])
    a.set_yticks([0, 5, 10])

ax01.set_title(r'$\text{A/B}$', size=19)
ax02.set_title(r'$\text{C}$', size=19)
ax01.set_xlabel(r'$\text{RNN 1}$', size=17)
ax01.set_ylabel(r'$\text{RNN 2}$', size=17)
ax02.set_xlabel(r'$\text{RNN 1}$', size=17)
ax02.set_ylabel(r'$\text{RNN 2}$', size=17)
ax01.legend(loc='lower right', frameon=False, fontsize=16, handletextpad=-0.2)

ax00.tick_params(axis='both', labelsize=16)
ax01.tick_params(axis='both', labelsize=16)
ax02.tick_params(axis='both', labelsize=16)

# =========================================================
# ------------- SECOND / THIRD ROW: STACKED ---------------
# =========================================================
subA = gs[1, 0].subgridspec(2, 1, hspace=0.35)
subB = gs[1, 1].subgridspec(2, 1, hspace=0.35)
subC = gs[1, 2].subgridspec(2, 1, hspace=0.35)

axA_in  = fig.add_subplot(subA[0])
axA_out = fig.add_subplot(subA[1], sharex=axA_in)

axB_in  = fig.add_subplot(subB[0])
axB_out = fig.add_subplot(subB[1], sharex=axB_in)

axC_in  = fig.add_subplot(subC[0])
axC_out = fig.add_subplot(subC[1], sharex=axC_in)

# =========================
# Task A
# =========================
sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_1[num_epochs-1, :]
student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

x, y, _ = flip_flop(
    t_max, dt, batch_size=1,
    stim_range=(1.0, 2.0),
    input_amp=2.0,
    target_amp=1.0,
    use_torch=True
)

t_ = dt * np.arange(x.shape[1])
B, T, _ = x.shape

axA_in.plot(t_, x[0, :, 0].detach().numpy(), lw=2)
if x.shape[-1] > 1:
    axA_in.plot(t_, x[0, :, 1].detach().numpy(), lw=2)

h = torch.zeros(2, 1)
y_pred = []
for t in range(T):
    x_t = x[:, t, :].T
    y_t, h = student_rnn_1(x_t, h)
    y_pred.append(y_t.T.unsqueeze(1))
y_pred = torch.cat(y_pred, dim=1)

axA_out.plot(t_, y[0, :, 0].detach().numpy(), lw=2, label=r'$\text{Target}$')
axA_out.plot(t_, y_pred[0, :, 0].detach().numpy(), '--k', label=r'$\text{RNN 1}$')
# axA_out.legend(loc='best', frameon=False, fontsize=16)

# =========================
# Task B
# =========================
sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_2[num_epochs-1, :]
student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

test_coherences = [-16, -8, -2, 2, 8, 16]
norm = plt.Normalize(vmin=-16, vmax=16)
cmap = cm.RdBu

for coh in test_coherences:
    x, y, _, _ = decision_making(
        t_max, dt, 1,
        coherences=[coh],
        noise_std=0.03,
        target_amp=1.0,
        fixed_coh_max=16.0,
        use_torch=True
    )

    color = cmap(norm(coh))
    t_dm = dt * np.arange(x.shape[1])

    axB_in.plot(t_dm, x[0, :, 0].detach().numpy(), lw=2, color=color)
    if x.shape[-1] > 1:
        axB_in.plot(t_dm, x[0, :, 1].detach().numpy(), lw=2, color=color)

    T = x.shape[1]
    h1 = torch.zeros(2, 1)
    y_pred1 = []

    with torch.no_grad():
        for t in range(T):
            x_t = x[:, t, :].T
            yt1, h1 = student_rnn_2(x_t, h1)
            y_pred1.append(yt1)

    y_p1 = torch.cat(y_pred1, dim=1).squeeze().numpy()
    y_target = y[0, :, 0].numpy()

    axB_out.plot(t_dm, y_target, color=color, lw=2)
    axB_out.plot(t_dm, y_p1, color=color, ls='--')

axB_out.axhline(0, color='k', alpha=0.2, lw=1)
# axB_out.legend(
#     handles=[
#         Line2D([0], [0], lw=2),
#         Line2D([0], [0], linestyle='--', lw=1)
#     ],
#     labels=[r'$\text{Target}$', r'$\text{RNN 2}$'],
#     loc='upper left',
#     frameon=False,
#     fontsize=16
# )

# =========================
# Task C
# =========================
sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_1[-1, :]
student_rnn_1 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, sig_zv, vv, zz = params_rnn_2[-1, :]
student_rnn_2 = effective_rnn_from_scaler_erf(sig_zm, sig_zu, sig_vm, sig_vu, sig_mu, mm, uu, dt=dt)

x_b, _, _ = flip_flop(t_max, dt, 1, use_torch=True)
x = (noise_scale * torch.randn((T, 1))) + (pulse_scale * x_b[:, :, 0].T) + mean
t_c = dt * np.arange(T)

axC_in.plot(t_c, x.squeeze().detach().numpy(), lw=2)

h_teacher = torch.zeros(2, 1)
h_student = torch.zeros(2, 1)
h_student2 = torch.zeros(2, 1)

with torch.no_grad():
    y_target_list = []
    for step in range(T):
        y_hat_t, h_teacher = teacher_eff_rnn(x[step], h_teacher)
        y_target_list.append(y_hat_t)
    y_target = torch.stack(y_target_list).squeeze()

y_pred_list = []
for step in range(T):
    y_hat_s, h_student = student_rnn_1(x[step], h_student)
    y_pred_list.append(y_hat_s)
y_pred = torch.stack(y_pred_list).squeeze()

y_pred_list2 = []
for step in range(T):
    y_hat_s, h_student2 = student_rnn_2(x[step], h_student2)
    y_pred_list2.append(y_hat_s)
y_pred2 = torch.stack(y_pred_list2).squeeze()

axC_out.plot(t_c, y_target.detach().numpy(), lw=2, label=r'$\text{Target}$')
axC_out.plot(t_c, y_pred.detach().numpy(), color="k", ls='--', label=r'$\text{RNN 1}$')
axC_out.plot(t_c, y_pred2.detach().numpy(), color="#DF0024", ls='--', label=r'$\text{RNN 2}$')
# axC_out.legend(loc='upper right', frameon=False, fontsize=16)

# =========================================================
# ----------------------- LABELS --------------------------
# =========================================================
axA_in.set_title(r"$\text{Task A (flip-flop)}$", size=18)
axB_in.set_title(r"$\text{Task B (decision-making)}$", size=18)
axC_in.set_title(r"$\text{Task C (arbitrary teacher)}$", size=18)

for a in [axA_in, axB_in, axC_in]:
    a.set_ylabel(r"$\text{Input}$", labelpad=-10, size=17)
    a.tick_params(labelbottom=False)
    a.spines['bottom'].set_visible(False)

for a in [axA_out, axB_out, axC_out]:
    a.set_ylabel(r"$\text{Output}$", size=17)
    a.set_xlabel(r"$\text{Time (s)}$", labelpad=-10, size=17)
    a.set_xticks([0, 20])
    a.spines['top'].set_visible(False)

for a in [axA_in, axB_in, axC_in]:
    a.set_ylabel(r"$\text{Input}$", size=17)
    a.yaxis.set_label_coords(-0.05, 0.5)  # same x for all

for a in [axA_out, axB_out, axC_out]:
    a.set_ylabel(r"$\text{Output}$", size=17)
    a.yaxis.set_label_coords(-0.05, 0.5)

# coherence colorbar
norm_cb = Normalize(vmin=-16, vmax=16)
sm = ScalarMappable(norm=norm_cb, cmap=cm.RdBu)
sm.set_array([])

cax = inset_axes(
    axB_in,
    width="27%",
    height="6%",
    loc='lower left',
    bbox_to_anchor=(.55, 0.06, 1, 1),
    bbox_transform=axB_in.transAxes
)

# axA_in.set_yticks([-1,1])
# axA_in.set_yticks([-1,1])
# axA_in.set_yticks([-1,1])

axA_in.set_yticks([-2, 2])
axA_in.set_yticklabels([-1, 1],size=16)

axB_in.set_yticks([-0.5, 0.5])
axB_in.set_yticklabels([-1, 1],size=16)

axC_in.set_yticks([-4, 4])
axC_in.set_yticklabels([-1, 1],size=16)

axA_out.set_yticks([-1,1])
axB_out.set_yticks([-1,1])
axC_out.set_yticks([-1,1])


axA_out.tick_params(axis='both', labelsize=16)
axB_out.tick_params(axis='both', labelsize=16)
axC_out.tick_params(axis='both', labelsize=16)


cbar = plt.colorbar(sm, cax=cax, orientation='horizontal')
cbar.ax.tick_params(labelsize=12)
cbar.set_ticks([-16, 0, 16])
cbar.set_label(r"$\text{coherence}$", rotation=0, labelpad=-26, fontsize=12, va='center')



ax00.text(
    -0.05, 1.2, r'($\mathbf{a}$)',
    transform=ax00.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

ax01.text(
    -0.1, 1.1, r'($\mathbf{b}$)',
    transform=ax01.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

ax02.text(
    -0.1, 1.1, r'($\mathbf{c}$)',
    transform=ax02.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

axA_in.text(
    -0.05, 1.5, r'($\mathbf{d}$)',
    transform=axA_in.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)


axB_in.text(
    -0.1, 1.5, r'($\mathbf{e}$)',
    transform=axB_in.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

axC_in.text(
    -0.16, 1.5, r'($\mathbf{f\,}$)',
    transform=axC_in.transAxes,
    fontsize=18, fontweight='bold',
    va='top', ha='right',
    font="Arial"
)

sns.despine()
plt.tight_layout()
# plt.savefig('../figures/fig_9.pdf')
plt.show()

In [ ]:
lab = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$',
       r'$\sigma_{mu}$', r'$||m||^2$', r'$||u||^2$',  r'$\sigma_{zv}$', 
       r'$||v||^2$', r'$||z||^2$']

fig, axes = plt.subplots(5, 2, figsize=(12, 8),sharex=True)

for i, ax in enumerate(axes.flat):

    color = "#0085C7" if i<7 else "#DF0024"  
    
    ax.plot(params_rnn_1[:, i],color=color, label=r'$\text{RNN 1}$',lw=2)
    ax.plot(params_rnn_2[:, i],color=color,ls='--', label=r'$\text{RNN 2}$',lw=2)
    ax.axhline(y=params_rnn_1[-1, i],ls='--',color='gray')
    ax.tick_params(labelsize=14)

    ax.yaxis.set_major_locator(MaxNLocator(nbins=1))

    # Calculate custom y-ticks: 5% padding from the limits
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    ticks = [y_min + 0.05 * y_range, y_max - 0.05 * y_range]
    
    # Set the ticks and the 2nd decimal precision formatter
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    # Set x-axis markers to 2000 and 6000 only
    ax.set_xticks([10000, 50000])

    ax.set_ylabel(lab[i],size=20)

    ax.axvline(x=30_000,color='k',lw=2, ymin=0.02,ymax=0.98,ls='--')

    if i == 0 :
        ax.legend(loc='lower right', frameon=False, fontsize=14, ncols=2, handletextpad=0.3)
        
    
axes[4,0].set_xlabel(r'$\text{Epoch}$',size=18)
axes[4,1].set_xlabel(r'$\text{Epoch}$',size=18)

axes[0,0].text(15_000,1.7,s='A/B')
axes[0,0].text(45_000,1.7,s='C')

axes[0,1].text(15_000,5.4,s='A/B')
axes[0,1].text(45_000,5.4,s='C')

plt.tight_layout()
sns.despine()
# plt.savefig('../figures/fig_10.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# --- Common Data ---
labels = [r'$\sigma_{zm}$', r'$\sigma_{zu}$', r'$\sigma_{vm}$', r'$\sigma_{vu}$',
          r'$\sigma_{mu}$', r'$\sigma_{zv}$', r'$||m||^2$', r'$||u||^2$', 
          r'$||v||^2$', r'$||z||^2$']

n = 10
C_BLUE = "#0085C7"
C_RED = "#DF0024"

# EXACT zeros based on your provided LaTeX matrix
matrix_zeros = [
    (3,0), (7,0), (8,0),          # Row 0
    (2,1), (6,1), (8,1),          # Row 1
    (1,2), (7,2), (9,2),          # Row 2
    (0,3), (6,3), (9,3),          # Row 3
    (5,4), (8,4), (9,4),          # Row 4
    (4,5), (6,5), (7,5),          # Row 5
    (1,6), (3,6), (5,6), (7,6), (8,6), (9,6), # Row 6
    (0,7), (2,7), (5,7), (6,7), (8,7), (9,7), # Row 7
    (0,8), (1,8), (4,8), (6,8), (7,8), (9,8), # Row 8
    (2,9), (3,9), (4,9), (6,9), (7,9), (8,9)  # Row 9
]

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5))

# ---------------------------------------------------------
# LEFT MATRIX: Original Circle Logic
# ---------------------------------------------------------
ax1.set(xlim=(-2, n), ylim=(0, n), aspect='equal')
ax1.invert_yaxis()
ax1.axis('off')

for i in range(n):
    for j in range(n):
        if i < 4:
            if j < 4: bg, cc, _a1, _a2 = C_BLUE, C_RED, 0.5, 0.9
            else: bg, cc, _a1, _a2 = C_RED, C_BLUE, 0.5, 0.9
        else:
            bg, _a1 = 'black', 0.1
            cc, _a2 = (C_BLUE if j < 4 else C_RED), 0.1
        
        if (i, j) in matrix_zeros:
            cc, _a2 = 'black', 0.0

        ax1.add_patch(Rectangle((i, j), 1, 1, fc=bg, ec='k', alpha=_a1))
        ax1.add_patch(Circle((i + .5, j + .5), .2, fc=cc, ec='k', alpha=_a2))

for j in range(n):
    ax1.text(-0.2, j + 0.5, labels[j], va='center', ha='right', fontsize=20, color=(C_BLUE if j < 4 else C_RED))

top_labels1 = [r'$\nabla_{\sigma_{zm}}$', r'$\nabla_{\sigma_{zu}}$', r'$\nabla_{\sigma_{vm}}$', r'$\nabla_{\sigma_{vu}}$',
               r'$0$', r'$0$', r'$0$', r'$0$', r'$0$', r'$0$']
for i in range(n):
    ax1.text(i + 0.5, -.4, top_labels1[i], ha='center', va='center', fontsize=20)
ax1.text(n/2, n/2, r'$\bar{G}$', ha='center', va='center', fontsize=100, color='k', alpha=.4)

# ---------------------------------------------------------
# RIGHT MATRIX: Wedge + Adaptive Alpha Logic
# ---------------------------------------------------------
ax2.set(xlim=(-2, n), ylim=(0, n), aspect='equal')
ax2.invert_yaxis()
ax2.axis('off')

dual_color = [(0,0), (1,1), (2,2), (3,3)] 

def get_entry_color_right(i, j):
    # Specific Red overrides: (5,5) and requested (8,8), (9,9)
    if (i, j) in [(5, 5), (8, 8), (9, 9)]:
        return C_RED
    
    # Requirement: (4,2), (2,4) AND (4,4) should be BLUE
    if (i, j) in [(4, 2), (2, 4), (4, 4)]:
        return C_BLUE

    # Red coordinates: sigma_zv positions
    red_coords = [(2,0), (0,2), (3,1), (1,3), (5,8), (8,5), (5,9), (9,5)]
    if (i, j) in red_coords:
        return C_RED
    
    # Default for bottom-right area (excluding the specific red diagonals handled above)
    if i >= 6 and j >= 6:
        return C_BLUE

    return C_BLUE

for i in range(n):
    for j in range(n):
        # Background logic
        if i < 5: bg, _a1 = (C_BLUE if j < 5 else C_RED), 0.5
        else: bg, _a1 = "black", 0.1
        
        if (i == 5 and j in (6, 7)) or (i > 7): bg, _a1 = "black", 0.1
        elif (i in (6, 7)) or (j in (6, 7)): bg, _a1 = C_BLUE, 0.5
        if i in (6,7) and j in (5,8,9): bg, _a1 = C_RED, 0.5

        ax2.add_patch(Rectangle((i, j), 1, 1, fc=bg, ec='k', alpha=_a1))

        # Circle/Wedge logic
        _a2_adaptive = 0.1 if bg == "black" else 0.9
        
        if (i, j) in matrix_zeros:
            continue
        elif (i, j) in dual_color:
            ax2.add_patch(Wedge((i + .5, j + .5), .2, 45, 225, fc=C_BLUE, ec='k', alpha=_a2_adaptive))
            ax2.add_patch(Wedge((i + .5, j + .5), .2, 225, 45, fc=C_RED, ec='k', alpha=_a2_adaptive))
        else:
            cc_right = get_entry_color_right(i, j)
            ax2.add_patch(Circle((i + .5, j + .5), .2, fc=cc_right, ec='k', alpha=_a2_adaptive))

for j in range(n):
    lab_color = C_BLUE if (j < 5 or j in (6, 7)) else C_RED
    ax2.text(-0.2, j + 0.5, labels[j], va='center', ha='right', fontsize=20, color=lab_color)

top_labels2 = [r'$\nabla_{\sigma_{zm}}$', r'$\nabla_{\sigma_{zu}}$', r'$\nabla_{\sigma_{vm}}$', r'$\nabla_{\sigma_{vu}}$',
               r'$\nabla_{\sigma_{mu}}$', r'$0$', r'$\nabla_{|m|^2}$', r'$\nabla_{|u|^2}$', r'$0$', r'$0$']
for i in range(n):
    ax2.text(i + 0.5, -.4, top_labels2[i], ha='center', va='center', fontsize=20)
ax2.text(n/2, n/2, r'$\bar{G}$', ha='center', va='center', fontsize=100, color='k', alpha=.4)

ax1.text(x=3.7,y=-1,s=r'$\text{Linear}$',size=22)
ax2.text(x=3.7,y=-1,s=r'$\text{Nonlinear}$',size=22)
plt.tight_layout()
# plt.savefig("../figures/fig_11.pdf", bbox_inches='tight')
plt.show()

In [ ]:
N = 500
dt = 0.05
t_max = 20
time_window = int(t_max/dt)
lr = 5e-3
c_star = 0.3
omega_star = 2
num_epochs = 10_000

# # ---------- TRAIN ----------
# m,u,v,z = initialize_vectors(N, rank=2)
# m = normalize(m) * np.sqrt(N)
# u = normalize(u) * np.sqrt(N)
# v = normalize(v) * np.sqrt(N)
# z = normalize(z) * np.sqrt(N)

# loss, params = train_model_rank_2( c_star, omega_star, N, num_epochs, time_window=time_window, m0=m, u0=u, v0=v, z0=z, dt=dt, lr=lr*N, plot_target=1)
# all_full_rnn = vectors_to_overlaps_rank_2(N,params,num_epochs)
# all_loss_21d, all_21d_rnn = run_21d_ode(params[0], N, dt, c_star, omega_star, num_epochs, time_window, lr, None)

# # ---------- SAVE ----------
# data_to_save = {
#     "loss": loss,
#     "all_loss_21d": all_loss_21d,
#     "all_full_rnn": all_full_rnn,
#     "all_21d_rnn": all_21d_rnn,
# }

# with open("../data/fig_12.pkl", "wb") as f:
#     pickle.dump(data_to_save, f)
# print("Saved successfully")

# ---------- LOAD ----------
with open("../data/fig_12.pkl", "rb") as f:
    loaded_data = pickle.load(f)

loss = loaded_data["loss"]
all_loss_21d = loaded_data["all_loss_21d"]
all_full_rnn = loaded_data["all_full_rnn"]
all_21d_rnn = loaded_data["all_21d_rnn"]
print("Loaded successfully fig_12")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax = ax.flatten()

# --- Panel 1: impulse response ---
(   sig_zm, sig_zu1, sig_zu2, sig_v1m, sig_v1u1, 
    sig_v1u2, sig_v2m, sig_v2u1, sig_v2u2, sig_mu1, 
    sig_mu2, sig_zv1, sig_zv2, sig_u1u2, sig_v1v2, 
    mm, uu1, uu2, vv1, vv2, zz ) = all_full_rnn[-1]

eff_rnn = effective_rnn_rank2_from_scalars(sig_zm, sig_zu1, sig_zu2, sig_v1m, sig_v1u1, sig_v1u2, sig_v2m, sig_v2u1, sig_v2u2, dt)

h = torch.zeros((3,1))               
x = torch.zeros((time_window,1))
x[0,0] = 1.0/dt   

t_axis = torch.linspace(0.0, float(time_window), time_window) * dt
y_target = np.exp(-c_star * t_axis) * np.cos(omega_star * t_axis)
             
y_pred = []  
for step in range(time_window):
    x_t = x[step, :].unsqueeze(0)
    y_hat, h = eff_rnn(x_t, h)
    y_pred.append(y_hat)

t_ = np.arange(time_window)*dt
ax[0].plot(t_, y_target, lw=3, alpha=1, label=r'$\text{Target filter}$')
ax[0].plot(t_, torch.stack(y_pred).flatten().detach(), '--k', label=r'$\text{RNN filter}$')

ax[0].set_yticks([-.6,  .8])
ax[0].set_xticks([0,  20])
ax[0].set_title(r'$\text{Impulse response}$', size=22)
ax[0].set_xlabel(r'$\text{Time (s)}$', size=18, labelpad=-15)
ax[0].set_ylabel(r'$\text{Output}$', size=18,labelpad=-25)
ax[0].legend(loc='upper right', frameon=False, fontsize=18, ncols=1)


ax[1].plot(np.array(loss), lw=3, label=r'$\text{Simulation}$')
ax[1].plot(np.array(all_loss_21d), '--k', label=r'$\text{Theory}\,\,\text{(ODE)}$')

ax[1].set_title(r'$\text{Loss curve}$', size=22)
ax[1].set_ylabel(r'$\text{Loss}$',size=18,labelpad=-15)
ax[1].set_yticks([0, .8])

ax[1].set_xlabel(r'$\text{Epoch}$', size=18, labelpad=-15)
# ax[1].set_xscale('log')
# ax[1].xaxis.set_minor_locator(NullLocator())
# ax[1].set_xticks([10, 1000])
ax[1].set_xticks([2000, 8000])

ax[1].legend(loc='lower left', frameon=False, fontsize=18)


lab = [
    r'$\sigma_{zm}$',     r'$\sigma_{z u_1}$', r'$\sigma_{z u_2}$',
    r'$\sigma_{v_1 m}$',  r'$\sigma_{v_1 u_1}$', r'$\sigma_{v_1 u_2}$',
    r'$\sigma_{v_2 m}$',  r'$\sigma_{v_2 u_1}$', r'$\sigma_{v_2 u_2}$'
]

cols = sns.color_palette('Blues_r', n_colors=9)
for i in range(9):
    ax[2].plot(np.array(all_full_rnn)[:,i], lw=3, color=cols[i], label=lab[i])
    ax[2].plot(np.array(all_21d_rnn)[:,i],lw=1,ls='--',color='k')


ax[2].set_title(r'$\text{Loss-visible}$', size=22)
ax[2].set_ylabel(r'$\text{Overlap}\,\,\, \boldsymbol{\sigma}$',size=18,labelpad=-15)
ax[2].set_yticks([-2, 2])

ax[2].set_xlabel(r'$\text{Epoch}$', size=18, labelpad=-15)
# ax[2].set_xscale('log')
# ax[2].xaxis.set_minor_locator(NullLocator())
# ax[2].set_xticks([10, 1000])
ax[2].set_xticks([2000, 8000])


lab = [
    r'$\sigma_{m u_1}$', r'$\sigma_{m u_2}$',
    r'$\sigma_{z v_1}$', r'$\sigma_{z v_2}$',
    r'$\sigma_{u_1 u_2}$', r'$\sigma_{v_1 v_2}$',
    r'$\|m\|^2$', r'$\|u_1\|^2$', r'$\|u_2\|^2$',
    r'$\|v_1\|^2$', r'$\|v_2\|^2$', r'$\|z\|^2$']


cols = sns.color_palette('Reds_r', n_colors=15)

for i in range(9,21):
    ax[3].plot(np.array(all_full_rnn)[:,i], lw=3, color=cols[i - 9], label=lab[i - 9])
    ax[3].plot(np.array(all_21d_rnn)[:,i],lw=1,ls='--',color='k')


ax[3].set_title(r'$\text{Loss-invisible}$', size=22)
ax[3].set_ylabel(r'$\text{Overlap}\,\,\, \tilde{\boldsymbol{\sigma}}$',size=18,labelpad=-15)
ax[3].set_yticks([-1, 2])

ax[3].set_xlabel(r'$\text{Epoch}$', size=18, labelpad=-15)
# ax[3].set_xscale('log')
# ax[3].xaxis.set_minor_locator(NullLocator())
# ax[3].set_xticks([10, 1000])
ax[3].set_xticks([2000, 8000])

# ax[2].legend(loc='upper left', frameon=False, fontsize=18, ncols=3)
# ax[3].legend(loc='center left', frameon=False, fontsize=18, ncols=3)

# === Subplot labels ===
subplot_labels = [r'($\mathbf{a}$)', r'($\mathbf{b}$)', r'($\mathbf{c}$)', r'($\mathbf{d}$)', ]
for i, label in enumerate(subplot_labels):
    ax[i].text(-0.05, 1.1, label, transform=ax[i].transAxes,
               fontsize=18, fontweight='bold', va='top', ha='right', font="Arial")


sns.despine()
plt.tight_layout()
plt.subplots_adjust(wspace=0.15)
plt.subplots_adjust(hspace=0.3)
# plt.savefig('../figures/fig_12.pdf', bbox_inches='tight')
plt.show()